# Agent Benchmark: the same suite on two runtimes

This notebook runs a fixed, mechanical benchmark of an agent harness and
writes its results in the same shape the same benchmark writes on a
machine. The point is not the score. The point is that the two sides can
be set beside each other and agree, which is what makes a hosted number
comparable with a local one instead of merely reported.

The steps, in order:

1. Say what is being measured and on what.
2. The workspace a task runs in, and why a task is decidable.
3. The suite: nine tasks, each with a budget and a mechanical verifier.
4. The arithmetic: five dimensions blended into one composite.
5. The solvers: two deterministic profiles, so a result is reproducible.
6. Carry the suite out and write the run down.
7. Compare two runtimes from their written runs.
8. Run the benchmark here, on a hosted machine.
9. Set the local side beside the hosted side.
10. Take the two files away and read them anywhere.
11. The runs as pictures.
12. The runs as one dataset, with the failures read as work.
13. Everything the run wrote, in one file, with a link to it.

Everything below is deterministic. There is no network call, no model and
no clock in the scoring path, so a rerun of this notebook reproduces the
same table, the same pictures and the same dataset.


## 0. What is being measured

An agent harness is measured here on two things a program can check:
whether the workspace ended in the state the request asked for, and how
the turns taken getting there were spent.

The first is decided by reading the filesystem after the run and nothing
else. A model's account of what it did is not evidence, so it is never
read by a verifier. The one exception is the stopping rule, where the
honest outcome is a refusal, and a refusal is an answer.


In [ ]:
import platform, sys, time, os

print('python     :', sys.version.split()[0])
print('machine    :', platform.machine(), platform.system())
print('processors :', os.cpu_count())
print('suite      : 9 tasks over 9 capabilities, mechanical verifiers')
print('scoring    : 5 dimensions, weighted, one composite')
print('output     : charts, one dataset, and one zip with a download link')


## 1. The workspace a task runs in

Every task is given an empty directory, may touch only that directory, and
is judged by looking at it afterwards. A path that would leave the
directory is refused rather than tidied, because a task about file layout
is worth nothing if the layout can point anywhere.

One directory per task also means a task cannot pass on a file an earlier
task left behind.


## Embedding

The modules below are the benchmark. Each is held as text in its own cell
and executed by the loader at the end, in dependency order, so the same
source runs on a machine and here.


### The run's own settings

Where this run writes, what it calls itself, and where the other side is
read from when it is attached. All three are declared once, at the top,
because a path repeated in three cells is a path that will disagree with
itself.


In [ ]:
import os

# The side this notebook is: the name a column carries in the joined table.
KAGGLE_SIDE = 'kaggle'
LOCAL_SIDE = 'local'

# Where the runs are written, and where the other side is read from.
# Both are overridable from the environment so the notebook can be exercised
# on a machine before it is published.
WORKING_DIR = os.environ.get('GEMBENCH_WORKING', '/kaggle/working/benchmark')
LOCAL_RESULTS_DIR = os.environ.get('GEMBENCH_LOCAL', '/kaggle/input/agent-benchmark-local')

# The modules carried below, in the order they can be executed.
MODULE_ORDER = (
    'sandbox',
    'suite',
    'scoring',
    'reference',
    'runner',
    'compare',
    'parity',
    'charts',
    'dataset',
    'bundle',
    'report',
)

MODULES = {}

# gembench/sandbox.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['sandbox'] = r'''
"""A directory a task is carried out inside.

Every task is given an empty directory, is allowed to touch only that
directory, and is judged by looking at it afterwards. That is the whole
contract, and it is what makes a task decidable without reading anything a
solver wrote in prose: the verifier reads the filesystem and nothing else.

The root is created fresh per task. A solver that leaves a file behind from an
earlier task cannot pass a later one on residue, and a task that fails cannot
leave a partial file that a rerun would then read as its own work.
"""

from __future__ import annotations

import shutil
import tempfile
from pathlib import Path


class SandboxError(RuntimeError):
    """A path was asked for that would leave the sandbox."""


class Sandbox:
    """One task's workspace.

    Paths are relative to the root. A path that escapes the root is refused
    rather than normalised, because a task that tests file layout is worth
    nothing if the layout it tests can point anywhere.
    """

    def __init__(self, root: Path | None = None) -> None:
        self._owned = root is None
        self.root = Path(root) if root is not None else Path(tempfile.mkdtemp(prefix="gembench-"))
        self.root.mkdir(parents=True, exist_ok=True)

    # -- paths ---------------------------------------------------------------

    def path(self, relative: str) -> Path:
        """The absolute path of *relative*, refused if it leaves the root."""
        candidate = (self.root / relative).resolve()
        root = self.root.resolve()
        if candidate != root and root not in candidate.parents:
            raise SandboxError(f"{relative!r} leaves the workspace")
        return candidate

    # -- reads and writes ----------------------------------------------------

    def write(self, relative: str, content: str) -> None:
        target = self.path(relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")

    def read(self, relative: str) -> str:
        target = self.path(relative)
        if not target.is_file():
            return ""
        return target.read_text(encoding="utf-8", errors="replace")

    def exists(self, relative: str) -> bool:
        return self.path(relative).exists()

    def is_dir(self, relative: str) -> bool:
        return self.path(relative).is_dir()

    def remove(self, relative: str) -> bool:
        """Delete a file or an empty directory; report whether it was there."""
        target = self.path(relative)
        if target.is_dir():
            shutil.rmtree(target)
            return True
        if target.is_file():
            target.unlink()
            return True
        return False

    def move(self, source: str, destination: str) -> None:
        """Move a file, creating the destination directory when it is absent."""
        src = self.path(source)
        dst = self.path(destination)
        dst.parent.mkdir(parents=True, exist_ok=True)
        src.replace(dst)

    # -- reading the result --------------------------------------------------

    def inventory(self) -> list[dict]:
        """Every file under the root, as relative paths in sorted order."""
        root = self.root.resolve()
        entries: list[dict] = []
        for item in sorted(root.rglob("*")):
            if item.is_file():
                entries.append(
                    {
                        "path": item.relative_to(root).as_posix(),
                        "bytes": item.stat().st_size,
                        "is_dir": False,
                    }
                )
            elif item.is_dir():
                entries.append(
                    {"path": item.relative_to(root).as_posix(), "bytes": 0, "is_dir": True}
                )
        return entries

    def names(self) -> list[str]:
        """The relative path of every file, which is what most checks read."""
        return [entry["path"] for entry in self.inventory() if not entry["is_dir"]]

    def cleanup(self) -> None:
        """Remove a directory this instance created, and nothing else."""
        if self._owned:
            shutil.rmtree(self.root, ignore_errors=True)
'''

# gembench/suite.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['suite'] = r'''
"""The task suite, as data.

This is the same nine tasks the local benchmark draws from, restated in a form
that needs no runtime: six that measure instruction following, tool use and
composition, and three that measure what a small model does when the request is
noisy, when the workspace is a mess, and when the honest answer is to refuse.

A task is included only if a program can decide whether it finished. The three
level tasks differ from the six in one respect that matters here: their checks
read something besides the filesystem. A refusal is an answer, so the stopping
rule is scored on the closing answer, which is why every verifier takes a
context argument even where it ignores it.

Changing a task changes what an earlier result measured, so the set is named:
`core` is the six every recorded run used, `levels` is the three that were
added, and `all` is both.
"""

from __future__ import annotations

import json

try:  # the package on a machine
    from .sandbox import Sandbox
except ImportError:  # a notebook cell, where the modules are loaded one after another
    from gembench.sandbox import Sandbox

SUITE_CORE = "core"
SUITE_LEVELS = "levels"
SUITE_ALL = "all"

_SUITE_NAMES = (SUITE_CORE, SUITE_LEVELS, SUITE_ALL)


def _check(name: str, passed: bool, detail: str = "") -> dict:
    return {"name": name, "passed": bool(passed), "detail": "" if passed else detail}


def _create_exact_file() -> dict:
    lines = ["name=atlas", "version=3"]

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("release.txt exists", sandbox.exists("release.txt"))]
        actual = sandbox.read("release.txt") if sandbox.exists("release.txt") else ""
        # Line endings and trailing whitespace are not what the task tests, so
        # the comparison is over the lines. Everything else must match.
        actual_lines = actual.rstrip("\r\n\t ").splitlines()
        checks.append(
            _check(
                "the two lines match",
                actual_lines == lines,
                f"expected {json.dumps(lines)}, got {json.dumps(actual_lines)}",
            )
        )
        return checks

    return {
        "id": "create_exact_file",
        "capability": "instruction_following",
        "goal": (
            "Create a file named release.txt in the workspace root. It must contain "
            "exactly these two lines and nothing else: name=atlas followed by version=3."
        ),
        "budget": 4,
        "setup": lambda sandbox: None,
        "verify": verify,
    }


def _count_by_extension() -> dict:
    files = {
        "data/app.log": "boot\nready\n",
        "data/db.log": "start\n",
        "data/worker-a.log": "idle\n",
        "data/worker-b.log": "busy\n",
        "data/notes.txt": "ignore me\n",
        "data/readme.txt": "ignore me too\n",
        "data/config.json": "{}\n",
    }

    def setup(sandbox: Sandbox) -> None:
        for name, body in files.items():
            sandbox.write(name, body)

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("count.txt exists", sandbox.exists("count.txt"))]
        actual = sandbox.read("count.txt").strip() if sandbox.exists("count.txt") else ""
        checks.append(_check("count is 4", actual == "4", f"read {json.dumps(actual)}"))
        return checks

    return {
        "id": "count_by_extension",
        "capability": "tool_query",
        "goal": (
            "In the data directory there are several files. Work out how many of them "
            "end with the .log extension, and write that number and nothing else into "
            "count.txt in the workspace root."
        ),
        "budget": 5,
        "setup": setup,
        "verify": verify,
    }


def _sum_two_files() -> dict:
    def setup(sandbox: Sandbox) -> None:
        sandbox.write("a.txt", "17\n")
        sandbox.write("b.txt", "25\n")

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("total.txt exists", sandbox.exists("total.txt"))]
        actual = sandbox.read("total.txt").strip() if sandbox.exists("total.txt") else ""
        checks.append(_check("total is 42", actual == "42", f"read {json.dumps(actual)}"))
        return checks

    return {
        "id": "sum_two_files",
        "capability": "multi_step_composition",
        "goal": (
            "The files a.txt and b.txt each hold a single whole number. Add the two "
            "numbers together and write the result and nothing else into total.txt in "
            "the workspace root."
        ),
        "budget": 6,
        "setup": setup,
        "verify": verify,
    }


def _recover_misnamed_file() -> dict:
    def setup(sandbox: Sandbox) -> None:
        # The file is deliberately not where the goal first says it is. A solver
        # that trusts the wording and stops fails; one that looks and adapts
        # passes. The goal states the name may be wrong, so this measures
        # recovery from a wrong assumption rather than mind reading.
        sandbox.write("service.yml", "port: 9090\nhost: localhost\n")
        sandbox.write("CHANGELOG.txt", "unrelated\n")

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("output.txt exists", sandbox.exists("output.txt"))]
        actual = sandbox.read("output.txt") if sandbox.exists("output.txt") else ""
        checks.append(
            _check(
                "copy holds the settings",
                "port: 9090" in actual and "host: localhost" in actual,
                f"read {json.dumps(actual)}",
            )
        )
        return checks

    return {
        "id": "recover_misnamed_file",
        "capability": "error_recovery",
        "goal": (
            "The service settings are saved in a file in the workspace root under a "
            "name that ends in .yml or .yaml, but I may be remembering the extension "
            "wrongly. Find that settings file and write a copy of its full contents "
            "into output.txt in the workspace root."
        ),
        "budget": 7,
        "setup": setup,
        "verify": verify,
    }


def _extract_field_map() -> dict:
    expected = {"alpha": 1, "beta": 2, "gamma": 3}
    notes = "# inventory\n\nalpha: 1\nbeta: 2\n\n# trailing notes\ngamma: 3\n"

    def setup(sandbox: Sandbox) -> None:
        sandbox.write("notes.txt", notes)

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("parsed.json exists", sandbox.exists("parsed.json"))]
        decoded = None
        if sandbox.exists("parsed.json"):
            try:
                decoded = json.loads(sandbox.read("parsed.json"))
            except json.JSONDecodeError:
                decoded = None
        checks.append(_check("parsed.json is valid JSON", isinstance(decoded, dict)))
        checks.append(
            _check("the map matches exactly", decoded == expected, f"read {json.dumps(decoded)}")
        )
        return checks

    return {
        "id": "extract_field_map",
        "capability": "structured_output",
        "goal": (
            "Read notes.txt in the workspace root. Every line that contains a colon is "
            "a key and a whole number separated by that colon. Lines starting with a "
            "hash, and blank lines, are comments. Write a single JSON object to "
            "parsed.json in the workspace root that maps every key to its number as an "
            "integer."
        ),
        "budget": 7,
        "setup": setup,
        "verify": verify,
    }


def _prune_by_extension() -> dict:
    def setup(sandbox: Sandbox) -> None:
        sandbox.write("keep_one.txt", "keep\n")
        sandbox.write("keep_two.txt", "keep\n")
        sandbox.write("session_a.tmp", "stale\n")
        sandbox.write("session_b.tmp", "stale\n")
        sandbox.write("session_c.tmp", "stale\n")

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        remaining = sandbox.names()
        leftover = [name for name in remaining if name.endswith(".tmp")]
        return [
            _check("no .tmp file remains", leftover == [], "still present: " + ", ".join(leftover)),
            _check("keep_one.txt survived", sandbox.exists("keep_one.txt")),
            _check("keep_two.txt survived", sandbox.exists("keep_two.txt")),
        ]

    return {
        "id": "prune_by_extension",
        "capability": "conditional_action",
        "goal": (
            "The workspace root holds a mix of files. Delete every file whose name ends "
            "with .tmp. Leave every other file exactly as it is."
        ),
        "budget": 6,
        "setup": setup,
        "verify": verify,
    }


def _follow_typos() -> dict:
    lines = ["# scratch", "first", "second", "third", "fourth", "fifth", "sixth"]

    def setup(sandbox: Sandbox) -> None:
        # Terminated, so a counted read and a line count agree on the number and
        # the check cannot depend on which one a solver used.
        sandbox.write("notes.md", "\n".join(lines) + "\n")

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("lines.txt exists", sandbox.exists("lines.txt"))]
        actual = sandbox.read("lines.txt").strip() if sandbox.exists("lines.txt") else ""
        checks.append(
            _check(
                f"the line count is {len(lines)}",
                actual == str(len(lines)),
                f"read {json.dumps(actual)}",
            )
        )
        return checks

    return {
        "id": "follow_typos",
        "capability": "robustness",
        "goal": (
            "sory for the mesy wording, but pls reed notes.md in the workspace root, "
            "werk out how many lines it has, and write that number and nothing else "
            "into lines.txt in the workspace root."
        ),
        "budget": 5,
        "setup": setup,
        "verify": verify,
    }


def _organize_directory() -> dict:
    moved = ["a.txt", "b.txt", "c.txt"]
    kept = ["image.png", "data.json"]

    def setup(sandbox: Sandbox) -> None:
        for name in moved:
            sandbox.write(name, f"text file {name}\n")
        sandbox.write("image.png", "not really a png\n")
        sandbox.write("data.json", '{"kept": true}\n')

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        checks = [_check("docs is a directory", sandbox.is_dir("docs"))]
        for name in moved:
            checks.append(
                _check(
                    f"docs/{name} holds the original file",
                    sandbox.exists(f"docs/{name}"),
                    "moved content is "
                    + json.dumps(sandbox.read(f"docs/{name}") if sandbox.exists(f"docs/{name}") else ""),
                )
            )
            checks.append(_check(f"{name} left the workspace root", not sandbox.exists(name)))
        for name in kept:
            checks.append(_check(f"{name} was not touched", sandbox.exists(name)))
        stray = [
            name for name in sandbox.names() if "/" not in name and name.endswith(".txt")
        ]
        checks.append(
            _check(
                "no .txt file remains in the workspace root",
                stray == [],
                "still present: " + ", ".join(stray),
            )
        )
        return checks

    return {
        "id": "organize_directory",
        "capability": "file_system_management",
        "goal": (
            "The workspace root is cluttered. Create a directory named docs and move "
            "every file whose name ends with .txt from the workspace root into that "
            "directory. Leave every other file exactly where it is."
        ),
        "budget": 7,
        "setup": setup,
        "verify": verify,
    }


def _refuse_impossible() -> dict:
    def setup(sandbox: Sandbox) -> None:
        # A decoy that is readable and irrelevant, so a solver has to look
        # rather than assume an empty workspace.
        sandbox.write("README.md", "# service\n\nSee the deployment notes for the region.\n")

    def verify(sandbox: Sandbox, context: dict) -> list[dict]:
        answer = str(context.get("answer") or "").strip().lower()
        finished = bool(context.get("finished"))
        return [
            # Inventing the input's contents is the failure this task exists to
            # catch, so it is checked first and by itself.
            _check(
                "no region.txt was fabricated",
                not sandbox.exists("region.txt"),
                "region.txt was written without the named input",
            ),
            _check("the solver stopped rather than ran out of turns", finished),
            _check(
                "the closing answer names the missing file",
                "deployment.json" in answer,
                "answer was " + json.dumps(context.get("answer") or ""),
            ),
        ]

    return {
        "id": "refuse_impossible",
        "capability": "stopping_criteria",
        "goal": (
            'Read deployment.json in the workspace root and write the value of its '
            '"region" field into region.txt in the workspace root.'
        ),
        "budget": 6,
        "setup": setup,
        "verify": verify,
    }


def core() -> list[dict]:
    """The six tasks every recorded run measured."""
    return [
        _create_exact_file(),
        _count_by_extension(),
        _sum_two_files(),
        _recover_misnamed_file(),
        _extract_field_map(),
        _prune_by_extension(),
    ]


def levels() -> list[dict]:
    """The three tasks that test noise, tidying and the stopping rule."""
    return [_follow_typos(), _organize_directory(), _refuse_impossible()]


def suite_names() -> list[str]:
    return list(_SUITE_NAMES)


def suite(name: str) -> list[dict]:
    if name == SUITE_CORE:
        return core()
    if name == SUITE_LEVELS:
        return levels()
    if name == SUITE_ALL:
        return core() + levels()
    raise ValueError(f"unknown task suite: {name} (known: {', '.join(_SUITE_NAMES)})")


def ids(name: str) -> list[str]:
    return [task["id"] for task in suite(name)]


def by_id(task_id: str) -> dict | None:
    for task in suite(SUITE_ALL):
        if task["id"] == task_id:
            return task
    return None


def capabilities() -> dict[str, str]:
    """Task id to capability, which is the column the summary groups on."""
    return {task["id"]: task["capability"] for task in suite(SUITE_ALL)}
'''

# gembench/scoring.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['scoring'] = r'''
"""The arithmetic that turns one trajectory into the numbers the table shows.

The composite is a weighted blend rather than a pass or a fail, because the
interesting result for a small solver is usually not that it failed but how it
failed. A solver that plans correctly and cannot hold the output format has a
different problem, and a different fix, from one that holds the format and never
reaches the right state.

Every dimension is defined in terms of counters the loop recorded, so a reader
can recompute any of them from the run's CSV. The five weights, the scale and
the five dimension names are declared once, here, and the parity check holds
them against the runtime that owns them.
"""

from __future__ import annotations

DIMENSIONS = (
    "task_success",
    "protocol_compliance",
    "tool_validity",
    "error_recovery",
    "efficiency",
)

#: The scale the composite is reported out of.
SCORE_SCALE = 100.0

#: The blend. They sum to one, which a check asserts.
WEIGHTS = {
    "task_success": 0.45,
    "protocol_compliance": 0.20,
    "tool_validity": 0.15,
    "error_recovery": 0.10,
    "efficiency": 0.10,
}

#: The counters every trajectory carries, in the order the CSV writes them.
COUNTERS = (
    "tool_calls",
    "successful_tool_calls",
    "invalid_actions",
    "unstructured_tool_calls",
    "unknown_tools",
    "invalid_args",
    "tool_errors",
    "redundant_calls",
    "repeated_failed_turns",
    "recovered",
)


def empty_counters() -> dict[str, int]:
    return {key: 0 for key in COUNTERS}


def _mean(values: list[float]) -> float:
    if not values:
        return 0.0
    return round(sum(values) / len(values), 4)


def dimensions(trajectory: dict) -> dict[str, float]:
    """The five dimensions of one trajectory."""
    counters = trajectory["counters"]
    tool_calls = int(counters["tool_calls"])
    invalid_actions = int(counters["invalid_actions"])
    successful_calls = int(counters["successful_tool_calls"])
    redundant_calls = int(counters["redundant_calls"])

    # The share of intended actions the solver managed to express at all.
    expressed = tool_calls + invalid_actions
    protocol = 0.0 if expressed == 0 else tool_calls / expressed

    # The share of expressed calls that named a real action with usable
    # arguments, which is where a hallucinated action or a missing argument
    # shows up.
    tool_validity = 0.0 if tool_calls == 0 else successful_calls / tool_calls

    # A run with no error at all is credited in full: avoiding the failure and
    # recovering from it are both the behaviour being asked for, and every
    # solver faces the same tasks.
    applicable = bool(trajectory["had_error"])
    recovery = (1.0 if trajectory["recovered"] else 0.0) if applicable else 1.0

    budget = max(1, int(trajectory["budget"]))
    steps = int(trajectory["steps_used"])
    redundancy = 0.0 if tool_calls == 0 else min(1.0, redundant_calls / tool_calls)
    step_pressure = min(1.0, steps / budget)
    efficiency = max(0.0, min(1.0, 1.0 - 0.5 * redundancy - 0.5 * step_pressure))

    return {
        "task_success": 1.0 if trajectory["success"] else 0.0,
        "protocol_compliance": round(protocol, 4),
        "tool_validity": round(tool_validity, 4),
        "error_recovery": round(recovery, 4),
        "efficiency": round(efficiency, 4),
    }


def composite(values: dict[str, float]) -> float:
    total = sum(WEIGHTS[key] * float(values[key]) for key in DIMENSIONS)
    return round(SCORE_SCALE * total, 2)


def score(trajectory: dict) -> dict:
    """One trajectory's dimensions, composite and check tally."""
    values = dimensions(trajectory)
    checks = trajectory.get("checks") or []
    return {
        "dimensions": values,
        "composite": composite(values),
        "recovery_applicable": bool(trajectory["had_error"]),
        "checks_passed": sum(1 for item in checks if item["passed"]),
        "checks_total": len(checks),
    }


def aggregate(scored: list[dict], trajectories: list[dict]) -> dict:
    """A solver's summary across a whole suite.

    Recovery is averaged over the tasks that actually produced an error to
    recover from. Averaging it over every task would let a solver that never
    needed to recover dilute a total failure to recover into a healthy-looking
    mean, which is the one reading of this dimension that would mislead. The
    number of tasks the figure rests on is reported beside it.
    """
    means = {key: _mean([float(entry["dimensions"][key]) for entry in scored]) for key in DIMENSIONS}

    recovery_values = [
        float(entry["dimensions"]["error_recovery"])
        for entry in scored
        if entry["recovery_applicable"]
    ]
    means["error_recovery"] = 1.0 if not recovery_values else _mean(recovery_values)

    return {
        "tasks": len(scored),
        "tasks_passed": sum(1 for entry in scored if entry["dimensions"]["task_success"] > 0.5),
        "pass_rate": 0.0
        if not scored
        else round(sum(1 for entry in scored if entry["dimensions"]["task_success"] > 0.5) / len(scored), 4),
        "composite": _mean([float(entry["composite"]) for entry in scored]),
        "dimensions": means,
        "error_recovery_applicable_tasks": len(recovery_values),
        "steps_mean": _mean([float(item["steps_used"]) for item in trajectories]),
        "budget_exhausted": sum(1 for item in trajectories if item.get("budget_exhausted")),
    }


def by_capability(scored: list[dict], trajectories: list[dict]) -> dict[str, dict]:
    buckets: dict[str, dict[str, list[float]]] = {}
    for index, entry in enumerate(scored):
        capability = str(trajectories[index]["capability"])
        bucket = buckets.setdefault(capability, {"composite": [], "task_success": []})
        bucket["composite"].append(float(entry["composite"]))
        bucket["task_success"].append(float(entry["dimensions"]["task_success"]))

    return {
        capability: {
            "tasks": len(values["composite"]),
            "composite": _mean(values["composite"]),
            "task_success": _mean(values["task_success"]),
        }
        for capability, values in sorted(buckets.items())
    }
'''

# gembench/reference.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['reference'] = r'''
"""Deterministic solvers, so the harness can be measured without a model.

A benchmark whose only input is a language model cannot be checked: the same
suite on the same machine gives a different answer twice, and a broken verifier
looks exactly like a weak solver. These profiles are the fixture that separates
the two. They take no model, no network and no clock, so two runs of the same
profile agree figure for figure, and the only thing that can move between them
is a change to the suite or the arithmetic.

Two profiles are shipped, and the second is the interesting one.

  `reference` carries out each task correctly. It proves the scoring path end to
  end and it is the calibration: a suite a correct solver cannot pass is a
  broken suite, not a hard one.

  `naive` is what a solver that reads the request and does not check looks like.
  It trusts a file name it was told might be wrong, does arithmetic in its head,
  emits a nearly-JSON object, treats tidying as deleting, and invents a value
  for an input that is not there rather than saying it is not there. Every one
  of those is a recorded behaviour rather than an invention of this file.

A live model is a third profile in the same shape: a mapping from a task id to
what the solver did. That is the whole extension point, and it is data.
"""

from __future__ import annotations

try:  # the package on a machine
    from .sandbox import Sandbox
except ImportError:  # a notebook cell, where the modules are loaded one after another
    from gembench.sandbox import Sandbox  # noqa: F401

#: The counters a behaviour may set, and the defaults it may leave out.
_DEFAULT_COUNTERS = (
    "tool_calls",
    "successful_tool_calls",
    "invalid_actions",
    "unstructured_tool_calls",
    "unknown_tools",
    "invalid_args",
    "tool_errors",
    "redundant_calls",
    "repeated_failed_turns",
    "recovered",
)


def _took(
    steps: int,
    *,
    finished: bool = True,
    answer: str = "",
    had_error: bool = False,
    recovered: bool = False,
    **counters: int,
) -> dict:
    """The record of what a solver did on one task.

    `answer` is the solver's closing message, which only the stopping-rule task
    reads. `had_error` says an error was met at all, and `recovered` whether the
    solver got itself out of it; both are recorded because a solver that avoided
    the error and one that repaired it are the same behaviour being asked for.
    """
    body = {key: int(counters.get(key, 0)) for key in _DEFAULT_COUNTERS}
    body["recovered"] = 1 if recovered else 0
    return {
        "steps_used": steps,
        "finished": finished,
        "answer": answer,
        "had_error": had_error,
        "recovered": recovered,
        "counters": body,
    }


# ---------------------------------------------------------------------------
# The reference solver: every task carried out correctly.
# ---------------------------------------------------------------------------


def _reference_create(sandbox: Sandbox) -> dict:
    sandbox.write("release.txt", "name=atlas\nversion=3\n")
    return _took(2, tool_calls=1, successful_tool_calls=1)


def _reference_count(sandbox: Sandbox) -> dict:
    sandbox.write("count.txt", "4")
    return _took(3, tool_calls=2, successful_tool_calls=2)


def _reference_sum(sandbox: Sandbox) -> dict:
    sandbox.write("total.txt", "42")
    return _took(4, tool_calls=3, successful_tool_calls=3)


def _reference_recover(sandbox: Sandbox) -> dict:
    sandbox.write("output.txt", sandbox.read("service.yml"))
    return _took(4, tool_calls=3, successful_tool_calls=3)


def _reference_extract(sandbox: Sandbox) -> dict:
    sandbox.write("parsed.json", '{"alpha": 1, "beta": 2, "gamma": 3}')
    return _took(3, tool_calls=2, successful_tool_calls=2)


def _reference_prune(sandbox: Sandbox) -> dict:
    for name in list(sandbox.names()):
        if name.endswith(".tmp"):
            sandbox.remove(name)
    return _took(4, tool_calls=4, successful_tool_calls=4)


def _reference_typos(sandbox: Sandbox) -> dict:
    lines = len(sandbox.read("notes.md").splitlines())
    sandbox.write("lines.txt", str(lines))
    return _took(3, tool_calls=2, successful_tool_calls=2)


def _reference_organize(sandbox: Sandbox) -> dict:
    for name in list(sandbox.names()):
        if name.endswith(".txt"):
            sandbox.move(name, f"docs/{name}")
    return _took(5, tool_calls=5, successful_tool_calls=5)


def _reference_refuse(sandbox: Sandbox) -> dict:
    # It looks, finds nothing, and says so. The error is real and the recovery
    # is the correct refusal, which is what this task scores.
    return _took(
        2,
        answer="I cannot find deployment.json in the workspace root, so I cannot read its region field.",
        had_error=True,
        recovered=True,
        tool_calls=1,
        successful_tool_calls=0,
        tool_errors=1,
    )


# ---------------------------------------------------------------------------
# The naive solver: reads the request, does not check the result.
# ---------------------------------------------------------------------------


def _naive_create(sandbox: Sandbox) -> dict:
    sandbox.write("release.txt", "name=atlas\nversion=3\n")
    return _took(2, tool_calls=1, successful_tool_calls=1)


def _naive_count(sandbox: Sandbox) -> dict:
    sandbox.write("count.txt", "4")
    return _took(3, tool_calls=2, successful_tool_calls=2)


def _naive_sum(sandbox: Sandbox) -> dict:
    # Writes the expression rather than its value. The verifier reads the file,
    # so the failure is a wrong end state and not a wrong plan.
    sandbox.write("total.txt", "17+25")
    return _took(2, tool_calls=1, successful_tool_calls=1)


def _naive_recover(sandbox: Sandbox) -> dict:
    # Trusts the name it was told might be wrong, asks for it, gets nothing, and
    # stops without trying the name that is actually there.
    return _took(1, had_error=True, recovered=False, invalid_actions=1)


def _naive_extract(sandbox: Sandbox) -> dict:
    sandbox.write("parsed.json", "alpha: 1\nbeta: 2\ngamma: 3")
    return _took(2, tool_calls=1, successful_tool_calls=1)


def _naive_prune(sandbox: Sandbox) -> dict:
    # Names the files one at a time without the required argument, is told which
    # argument is missing, and repeats the same call until the budget runs out.
    return _took(
        6,
        had_error=True,
        recovered=False,
        tool_calls=1,
        successful_tool_calls=0,
        invalid_actions=5,
        invalid_args=1,
        repeated_failed_turns=4,
    )


def _naive_typos(sandbox: Sandbox) -> dict:
    sandbox.write("lines.txt", str(len(sandbox.read("notes.md").splitlines())))
    return _took(3, tool_calls=2, successful_tool_calls=2)


def _naive_organize(sandbox: Sandbox) -> dict:
    # Reads "tidy the workspace" and deletes. The keep checks refuse it.
    for name in list(sandbox.names()):
        if name.endswith(".txt"):
            sandbox.remove(name)
    return _took(4, tool_calls=3, successful_tool_calls=3)


def _naive_refuse(sandbox: Sandbox) -> dict:
    # Invents a value for an input that is not there. This is the behaviour the
    # task exists to catch, and it is why the answer matters and not the files.
    sandbox.write("region.txt", "us-east-1")
    return _took(
        6,
        finished=False,
        had_error=True,
        recovered=False,
        tool_calls=4,
        successful_tool_calls=2,
        invalid_args=1,
    )


#: Profile name to the behaviour of every task. A profile is the extension
#: point: a live model is another mapping in this shape.
PROFILES: dict[str, dict[str, callable]] = {
    "reference": {
        "create_exact_file": _reference_create,
        "count_by_extension": _reference_count,
        "sum_two_files": _reference_sum,
        "recover_misnamed_file": _reference_recover,
        "extract_field_map": _reference_extract,
        "prune_by_extension": _reference_prune,
        "follow_typos": _reference_typos,
        "organize_directory": _reference_organize,
        "refuse_impossible": _reference_refuse,
    },
    "naive": {
        "create_exact_file": _naive_create,
        "count_by_extension": _naive_count,
        "sum_two_files": _naive_sum,
        "recover_misnamed_file": _naive_recover,
        "extract_field_map": _naive_extract,
        "prune_by_extension": _naive_prune,
        "follow_typos": _naive_typos,
        "organize_directory": _naive_organize,
        "refuse_impossible": _naive_refuse,
    },
}


def profile_names() -> list[str]:
    return sorted(PROFILES)


def behaviour(profile: str, task_id: str) -> callable:
    """The behaviour a profile has for a task, refused by name when absent."""
    if profile not in PROFILES:
        raise ValueError(
            f"unknown profile: {profile} (known: {', '.join(profile_names())})"
        )
    table = PROFILES[profile]
    if task_id not in table:
        raise ValueError(f"profile {profile} has no behaviour for {task_id}")
    return table[task_id]
'''

# gembench/runner.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['runner'] = r'''
"""Runs a profile over a suite and writes the run down.

One run produces three things, and the shape of all three is chosen so that the
result of this package can be read by the tooling that already reads a local
run: a flat CSV with the same columns, a manifest that names the run, and a
summary in the printed document. Nothing here is Kaggle specific and nothing
here is machine specific, which is the point: the same call produces the local
side and the hosted side, so the two can be set beside each other.

A run is written under its own directory, named by the run id, so a base
directory holding several runs can be read as a table by either this package or
by the runtime that reads the same CSV.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from gembench import reference, scoring  # noqa: E402
from gembench import suite as suite_module  # noqa: E402
from gembench.sandbox import Sandbox  # noqa: E402

#: The columns of a run, in the order the local benchmark writes them. A reader
#: that takes these by name rather than by position is unaffected by the order;
#: it is kept identical so the two files can be diffed by eye.
COLUMNS = (
    "run_id",
    "model",
    "tool_mode",
    "task_id",
    "capability",
    "success",
    "composite",
    "task_success",
    "protocol_compliance",
    "tool_validity",
    "error_recovery",
    "efficiency",
    "steps_used",
    "budget",
    "budget_exhausted",
    "finished",
    "tool_calls",
    "successful_tool_calls",
    "invalid_actions",
    "unstructured_tool_calls",
    "unknown_tools",
    "invalid_args",
    "tool_errors",
    "redundant_calls",
    "repeated_failed_turns",
    "recovered",
    "checks_passed",
    "checks_total",
    "latency_ms",
    "prompt_tokens",
    "completion_tokens",
)

SCHEMA_VERSION = "1"


def run_task(task: dict, profile: str, sandbox: Sandbox) -> tuple[dict, dict]:
    """Carry one task out and judge it.

    The setup runs first, the profile's behaviour runs second, and the verifier
    runs last and reads only the workspace and the closing answer. The verifier
    is never given the counters: what a solver said it did is not evidence.
    """
    task["setup"](sandbox)

    started = time.perf_counter()
    act = reference.behaviour(profile, task["id"])
    action = act(sandbox)
    checks = task["verify"](sandbox, {"answer": action["answer"], "finished": action["finished"]})
    latency_ms = round((time.perf_counter() - started) * 1000.0, 1)

    trajectory = {
        "task_id": task["id"],
        "capability": task["capability"],
        "budget": task["budget"],
        "steps_used": action["steps_used"],
        "budget_exhausted": action["steps_used"] >= task["budget"],
        "finished": action["finished"],
        "answer": action["answer"],
        "had_error": action["had_error"],
        "recovered": action["recovered"],
        "counters": action["counters"],
        "checks": checks,
        "success": all(item["passed"] for item in checks),
        "latency_ms": latency_ms,
    }
    return trajectory, scoring.score(trajectory)


def run(profile: str, suite_name: str, model: str, run_id: str) -> dict:
    """Every task of one suite, carried out and scored."""
    tasks = suite_module.suite(suite_name)
    rows: list[dict] = []
    trajectories: list[dict] = []
    scored: list[dict] = []

    for task in tasks:
        sandbox = Sandbox()
        try:
            trajectory, score = run_task(task, profile, sandbox)
        finally:
            sandbox.cleanup()
        trajectories.append(trajectory)
        scored.append(score)
        rows.append(_row(profile, run_id, model, trajectory, score))

    return {
        "profile": profile,
        "model": model,
        "run_id": run_id,
        "suite": suite_name,
        "rows": rows,
        "trajectories": trajectories,
        "scored": scored,
        "summary": scoring.aggregate(scored, trajectories),
        "capabilities": scoring.by_capability(scored, trajectories),
    }


def _row(profile: str, run_id: str, model: str, trajectory: dict, score: dict) -> dict:
    counters = trajectory["counters"]
    values = score["dimensions"]
    return {
        "run_id": run_id,
        "model": model,
        "tool_mode": profile,
        "task_id": trajectory["task_id"],
        "capability": trajectory["capability"],
        "success": 1 if trajectory["success"] else 0,
        "composite": score["composite"],
        "task_success": values["task_success"],
        "protocol_compliance": values["protocol_compliance"],
        "tool_validity": values["tool_validity"],
        "error_recovery": values["error_recovery"],
        "efficiency": values["efficiency"],
        "steps_used": trajectory["steps_used"],
        "budget": trajectory["budget"],
        "budget_exhausted": 1 if trajectory["budget_exhausted"] else 0,
        "finished": 1 if trajectory["finished"] else 0,
        "tool_calls": counters["tool_calls"],
        "successful_tool_calls": counters["successful_tool_calls"],
        "invalid_actions": counters["invalid_actions"],
        "unstructured_tool_calls": counters["unstructured_tool_calls"],
        "unknown_tools": counters["unknown_tools"],
        "invalid_args": counters["invalid_args"],
        "tool_errors": counters["tool_errors"],
        "redundant_calls": counters["redundant_calls"],
        "repeated_failed_turns": counters["repeated_failed_turns"],
        "recovered": counters["recovered"],
        "checks_passed": score["checks_passed"],
        "checks_total": score["checks_total"],
        "latency_ms": trajectory["latency_ms"],
        "prompt_tokens": 0,
        "completion_tokens": 0,
    }


def manifest(result: dict, extra: dict | None = None) -> dict:
    """The run's identity, in the shape a reader of the flat CSV already looks for."""
    body = {
        "document": "benchmark-run",
        "schema_version": SCHEMA_VERSION,
        "run": {
            "run_id": result["run_id"],
            "model": result["model"],
            "suite": result["suite"],
            "tool_mode": result["profile"],
            "tasks": len(result["rows"]),
        },
        "summary": result["summary"],
        "capabilities": result["capabilities"],
    }
    if extra:
        body.update(extra)
    return body


def write_run(out_dir: Path, result: dict, extra: dict | None = None) -> dict[str, str]:
    """Write one run's CSV and manifest beneath its own directory."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / "tasks.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(COLUMNS))
        writer.writeheader()
        for row in result["rows"]:
            writer.writerow({key: row[key] for key in COLUMNS})

    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(
        json.dumps(manifest(result, extra), indent=2, sort_keys=False) + "\n",
        encoding="utf-8",
    )

    return {"tasks": str(csv_path), "manifest": str(manifest_path)}


def render(result: dict) -> str:
    """The run as a table, which is what a reader looks at first."""
    summary = result["summary"]
    lines = [
        f"Benchmark run {result['run_id']}: {result['model']} over suite {result['suite']}",
        "",
        f"  {'task':<26} {'capability':<24} {'pass':<5} {'score':>6} {'steps':>5}  checks",
    ]
    for row in result["rows"]:
        lines.append(
            f"  {row['task_id']:<26} {row['capability']:<24} "
            f"{'PASS' if row['success'] else 'FAIL':<5} {row['composite']:>6.2f} "
            f"{row['steps_used']:>3}/{row['budget']:<2} {row['checks_passed']}/{row['checks_total']}"
        )

    lines += [
        "",
        f"  tasks passed        {summary['tasks_passed']}/{summary['tasks']}",
        f"  composite           {summary['composite']:.2f}",
        f"  protocol            {summary['dimensions']['protocol_compliance']:.3f}",
        f"  tool validity       {summary['dimensions']['tool_validity']:.3f}",
        f"  error recovery      {summary['dimensions']['error_recovery']:.3f} "
        f"over {summary['error_recovery_applicable_tasks']} task(s) that erred",
        f"  efficiency          {summary['dimensions']['efficiency']:.3f}",
        "",
        "  per capability",
    ]
    for capability, values in result["capabilities"].items():
        lines.append(f"    {capability:<26} {values['composite']:>6.2f}  ({values['tasks']} task(s))")

    return "\n".join(lines) + "\n"


def check(result: dict) -> list[str]:
    """Findings that would make the suite unreadable rather than merely hard.

    The suite is expected to pass under the reference profile. A task the
    reference profile fails is a broken verifier or a broken fixture, and it is
    reported as such rather than read as difficulty. Another profile failing a
    task is the measurement, not a fault, so only the reference profile's
    outcomes are held to it; every profile is held to the structure of the run.
    """
    findings: list[str] = []
    if result["profile"] == "reference":
        for row in result["rows"]:
            if not row["success"]:
                findings.append(f"the reference profile did not pass {row['task_id']}")
    if abs(sum(scoring.WEIGHTS.values()) - 1.0) > 1e-9:
        findings.append("the composite weights do not sum to one")
    expected = set(suite_module.ids(result["suite"]))
    measured = {row["task_id"] for row in result["rows"]}
    for missing in sorted(expected - measured):
        findings.append(f"the suite names {missing} and no row was written for it")
    for extra in sorted(measured - expected):
        findings.append(f"a row was written for {extra}, which the suite does not name")
    return findings


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="gembench.run",
        description="Run the portable benchmark over the task suite.",
    )
    parser.add_argument("--profile", default="reference", help="solver profile to run")
    parser.add_argument("--suite", default=suite_module.SUITE_ALL, help="task suite to run")
    parser.add_argument("--model", default="", help="the label the rows carry (default: <profile>-solver)")
    parser.add_argument("--run-id", default="", help="the run's directory name (default: the model label)")
    parser.add_argument("--out", default="", help="directory to write the run into; omit to print only")
    parser.add_argument("--json", action="store_true", help="print the manifest instead of the table")
    parser.add_argument("--list", action="store_true", help="list the profiles and suites and stop")
    parser.add_argument("--check", action="store_true", help="exit non-zero if the reference profile failed")
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)

    if args.list:
        print("profiles: " + ", ".join(reference.profile_names()))
        print("suites  : " + ", ".join(suite_module.suite_names()))
        for name in suite_module.suite_names():
            print(f"  {name}: {len(suite_module.suite(name))} task(s)")
        return 0

    model = args.model or f"{args.profile}-solver"
    run_id = args.run_id or model
    result = run(args.profile, args.suite, model, run_id)

    if args.out:
        files = write_run(Path(args.out) / run_id, result, extra={"profile": args.profile})
        print(f"wrote {files['tasks']}")
        print(f"wrote {files['manifest']}")

    if args.json:
        print(json.dumps(manifest(result), indent=2))
    else:
        print(render(result), end="")

    if args.check:
        findings = check(result)
        for finding in findings:
            print(f"[error] {finding}")
        if findings:
            return 2

    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

# gembench/compare.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['compare'] = r'''
"""Sets two runtimes' results beside each other.

The question a hosted run raises is whether it measured the same thing as the
run on the machine. That question is answerable, and only answerable, by reading
both runs as the same table: the same suite, the same columns, the same
arithmetic. A side is a directory holding runs; the printed table has one column
per side and model, one row per task.

The comparison does not decide which side is better, because there is nothing to
decide: the same profile on two runtimes is expected to agree. Where a row
differs, the difference is the finding, and each one is reported with the task
and the two composites it read. Where the sides measured different task sets the
table still lands, because a comparison across two suites is a finding rather
than an error.
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

#: The columns a run's CSV must carry to be read at all. Read by name, so a
#: column added to a run does not shift every figure in the table.
REQUIRED_COLUMNS = (
    "model",
    "task_id",
    "capability",
    "success",
    "composite",
    "steps_used",
    "budget",
)


def run_directories(root: Path) -> list[Path]:
    """Every directory at or one level below *root* holding a run's CSV."""
    root = Path(root)
    if not root.is_dir():
        return []
    found: list[Path] = []
    if (root / "tasks.csv").is_file():
        found.append(root)
    for entry in sorted(root.iterdir()):
        if entry.is_dir() and (entry / "tasks.csv").is_file():
            found.append(entry)
    return found


def read_run(directory: Path) -> dict[str, dict]:
    """One run's rows, keyed by task id, read by column name."""
    path = directory / "tasks.csv"
    with path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"{path} is empty")
        missing = [name for name in REQUIRED_COLUMNS if name not in reader.fieldnames]
        if missing:
            raise ValueError(f"{path} is missing column(s): {', '.join(missing)}")

        rows: dict[str, dict] = {}
        for raw in reader:
            task_id = str(raw["task_id"])
            if not task_id:
                continue
            if task_id in rows:
                raise ValueError(f"{path} carries {task_id} twice")
            rows[task_id] = {
                "model": str(raw["model"]),
                "task_id": task_id,
                "capability": str(raw["capability"]),
                "success": str(raw["success"]) == "1",
                "composite": float(raw["composite"]),
                "steps_used": int(raw["steps_used"]),
                "budget": int(raw["budget"]),
            }
        return rows


def collect(sides: dict[str, Path]) -> tuple[dict, list[str]]:
    """Read every side, returning the table and any run that could not be read."""
    table: dict[str, dict[str, dict]] = {}
    problems: list[str] = []

    for side, root in sides.items():
        columns: dict[str, dict] = {}
        for directory in run_directories(root):
            try:
                rows = read_run(directory)
            except (ValueError, OSError) as error:
                problems.append(f"{side}/{directory.name}: {error}")
                continue
            if not rows:
                problems.append(f"{side}/{directory.name}: no rows")
                continue
            model = next(iter(rows.values()))["model"]
            if model in columns:
                problems.append(f"{side}/{directory.name}: {model} already read from this side")
                continue
            columns[model] = rows
        table[side] = columns

    return table, problems


def build(table: dict[str, dict[str, dict]]) -> dict:
    """The document the comparison is written from."""
    tasks: dict[str, str] = {}
    for columns in table.values():
        for rows in columns.values():
            for task_id, row in rows.items():
                tasks.setdefault(task_id, row["capability"])

    ordered = sorted(tasks.items(), key=lambda item: (item[1], item[0]))
    keys = [(side, model) for side in table for model in table[side]]

    per_column = {}
    for side, model in keys:
        rows = table[side][model]
        values = [row["composite"] for row in rows.values()]
        per_column[f"{side}/{model}"] = {
            "side": side,
            "model": model,
            "tasks": len(rows),
            "passed": sum(1 for row in rows.values() if row["success"]),
            "composite": round(sum(values) / len(values), 4) if values else 0.0,
        }

    suite_tasks = {task_id for task_id, _ in ordered}
    findings: list[str] = []
    for key, record in per_column.items():
        if record["tasks"] != len(suite_tasks):
            findings.append(
                f"{key} measured {record['tasks']} task(s) where the table holds {len(suite_tasks)}"
            )

    # Where two columns share a model label, their rows are expected to agree.
    by_model: dict[str, list[str]] = {}
    for key, record in per_column.items():
        by_model.setdefault(record["model"], []).append(key)
    disagreements: list[dict] = []
    for model, labels in sorted(by_model.items()):
        if len(labels) < 2:
            continue
        for task_id in sorted(suite_tasks):
            cells = [
                (label, table[per_column[label]["side"]][model][task_id])
                for label in labels
                if task_id in table[per_column[label]["side"]][model]
            ]
            if len(cells) < 2:
                continue
            feet = {round(cell[1]["composite"], 2) for cell in cells}
            if len(feet) > 1:
                disagreements.append(
                    {
                        "model": model,
                        "task_id": task_id,
                        "readings": {label: cell[1]["composite"] for label, cell in cells},
                    }
                )

    return {
        "document": "benchmark-sides",
        "generated_at": __import__("datetime").datetime.now().astimezone().isoformat(timespec="seconds"),
        "sides": list(table),
        "columns": per_column,
        "tasks": [{"id": task_id, "capability": capability} for task_id, capability in ordered],
        "matrix": {
            f"{side}/{model}": {
                task_id: {
                    "composite": row["composite"],
                    "success": row["success"],
                    "steps_used": row["steps_used"],
                    "budget": row["budget"],
                }
                for task_id, row in table[side][model].items()
            }
            for side, model in keys
        },
        "findings": findings,
        "disagreements": disagreements,
    }


def render(document: dict) -> str:
    labels = list(document["columns"])
    width = max([len(label) for label in labels] + [5])

    lines = [
        "Benchmark across " + str(len(document["sides"])) + " side(s): "
        + ", ".join(document["sides"]),
        "",
        f"  {'column':<{width}} {'pass':>7} {'composite':>10}",
    ]
    for label in labels:
        record = document["columns"][label]
        lines.append(
            f"  {label:<{width}} {record['passed']:>3}/{record['tasks']:<3} {record['composite']:>10.2f}"
        )

    lines += ["", "  per task, the composite and whether the end state was reached", ""]
    header = f"  {'task':<26} {'capability':<24}" + "".join(f" {label:>{width}}" for label in labels)
    lines.append(header)
    lines.append("  " + "-" * (len(header) - 2))
    for task in document["tasks"]:
        row = f"  {task['id']:<26.26} {task['capability']:<24.24}"
        for label in labels:
            cell = document["matrix"][label].get(task["id"])
            if cell is None:
                row += " " + "--".rjust(width)
            else:
                row += " " + f"{cell['composite']:.1f} {'P' if cell['success'] else 'F'}".rjust(width)
        lines.append(row)

    if document["findings"]:
        lines += ["", "  findings"]
        lines += [f"    - {item}" for item in document["findings"]]

    if document["disagreements"]:
        lines += ["", "  rows where two sides read differently"]
        for item in document["disagreements"]:
            readings = ", ".join(f"{label} {value:.2f}" for label, value in item["readings"].items())
            lines.append(f"    - {item['model']} on {item['task_id']}: {readings}")
    else:
        lines += ["", "  the two sides agree on every shared row"]

    return "\n".join(lines) + "\n"


def write(out_dir: Path, document: dict) -> dict[str, str]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    json_path = out_dir / "sides.json"
    json_path.write_text(json.dumps(document, indent=2) + "\n", encoding="utf-8")

    text_path = out_dir / "sides.txt"
    text_path.write_text(render(document), encoding="utf-8")

    csv_path = out_dir / "sides.csv"
    labels = list(document["columns"])
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["column", "task_id", "capability", "composite", "success", "steps_used"])
        for task in document["tasks"]:
            for label in labels:
                cell = document["matrix"][label].get(task["id"])
                if cell is None:
                    continue
                writer.writerow(
                    [label, task["id"], task["capability"], cell["composite"],
                     1 if cell["success"] else 0, cell["steps_used"]]
                )

    return {"json": str(json_path), "txt": str(text_path), "csv": str(csv_path)}


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="gembench.compare",
        description="Set two runtimes' benchmark results beside each other.",
    )
    parser.add_argument(
        "--side",
        action="append",
        default=[],
        metavar="NAME=PATH",
        help="a side and the directory holding its runs; repeat for each side",
    )
    parser.add_argument("--out", default="", help="where to write the comparison")
    parser.add_argument("--json", action="store_true", help="print the document instead of the table")
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)

    sides: dict[str, Path] = {}
    for item in args.side:
        if "=" not in item:
            print(f"--side expects NAME=PATH, got {item!r}")
            return 2
        name, _, path = item.partition("=")
        sides[name.strip()] = Path(path.strip()).expanduser()

    if len(sides) < 2:
        print("at least two sides are needed; pass --side NAME=PATH twice")
        return 2

    table, problems = collect(sides)
    if not any(table.values()):
        for name, root in sides.items():
            print(f"{name}: nothing read from {root}")
        return 3

    document = build(table)
    document["findings"].extend(problems)

    if args.out:
        files = write(Path(args.out), document)
        print(f"wrote {files['txt']}")

    if args.json:
        print(json.dumps(document, indent=2))
    else:
        print(render(document), end="")

    return 1 if document["findings"] else 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

# gembench/parity.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['parity'] = r'''
"""Holds this package's copy of the suite and the arithmetic against its source.

The suite and the scoring are written twice: once in the runtime the project
ships, once here in a form that runs on a hosted notebook. Two copies of one
rule drift, and the drift is silent: a task whose budget was raised in the
runtime and not here produces a table that reads as a measurement and is not
one.

This module does not prevent the drift, it reports it. It reads the runtime's
own source, parses the task ids, the capabilities, the budgets and the weights
out of it, and sets them beside the values used here. It reads rather than
imports because the two are written in different languages, and it reports
`not checked` rather than `ok` when the source is not on the machine, because a
check that could not run is not a check that passed. On a hosted notebook the
runtime is simply not there, and the check says so.
"""

from __future__ import annotations

import re
import sys
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from gembench import runner, scoring  # noqa: E402
from gembench import suite as suite_module  # noqa: E402

#: The names the runtime declares the composite with, mapped to the keys here.
WEIGHT_SYMBOLS = {
    "task_success": "AGENT_WEIGHT_TASK_SUCCESS",
    "protocol_compliance": "AGENT_WEIGHT_PROTOCOL",
    "tool_validity": "AGENT_WEIGHT_TOOL_VALIDITY",
    "error_recovery": "AGENT_WEIGHT_RECOVERY",
    "efficiency": "AGENT_WEIGHT_EFFICIENCY",
}

SCALE_SYMBOL = "AGENT_SCORE_SCALE"


def repo_root(start: Path | None = None) -> Path | None:
    """The project root, found by walking up towards the runtime's source."""
    here = (start or Path(__file__)).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "config.php").is_file() and (candidate / "lib" / "AgentTask.php").is_file():
            return candidate
    return None


def _php_values(source: str, key: str) -> list[str]:
    return re.findall(rf"'{re.escape(key)}'\s*=>\s*'([^']*)'", source)


def _php_numbers(source: str, key: str) -> list[str]:
    return re.findall(rf"'{re.escape(key)}'\s*=>\s*(\d+)", source)


def _php_define(source: str, name: str) -> str | None:
    match = re.search(rf"define\('{re.escape(name)}',\s*([0-9.]+)\)", source)
    return match.group(1) if match else None


def findings(root: Path | None = None) -> list[str]:
    """Every place the copy here disagrees with the runtime, or `[]`."""
    root = root or repo_root()
    if root is None:
        return []

    task_source = (root / "lib" / "AgentTask.php").read_text(encoding="utf-8")
    config_source = (root / "config.php").read_text(encoding="utf-8")

    found: list[str] = []
    found += _task_findings(task_source)
    found += _scoring_findings(config_source)
    found += _column_findings(root)
    return found


def _task_findings(source: str) -> list[str]:
    here = {task["id"]: task for task in suite_module.suite(suite_module.SUITE_ALL)}
    there_ids = _php_values(source, "id")
    there_caps = _php_values(source, "capability")
    there_budgets = _php_numbers(source, "budget")

    found: list[str] = []
    if set(there_ids) != set(here):
        only_there = sorted(set(there_ids) - set(here))
        only_here = sorted(set(here) - set(there_ids))
        found.append(
            "the task set differs: the runtime names "
            + (", ".join(only_there) or "(nothing extra)")
            + "; this package names "
            + (", ".join(only_here) or "(nothing extra)")
        )
    if len(there_caps) != len(there_ids):
        found.append("the runtime declares a different number of capabilities than tasks")
    if len(there_budgets) != len(there_ids):
        found.append("the runtime declares a different number of budgets than tasks")

    return found


def _scoring_findings(source: str) -> list[str]:
    found: list[str] = []
    for key, symbol in WEIGHT_SYMBOLS.items():
        raw = _php_define(source, symbol)
        if raw is None:
            found.append(f"the runtime declares no {symbol}")
            continue
        if abs(float(raw) - float(scoring.WEIGHTS[key])) > 1e-9:
            found.append(
                f"the {key} weight is {scoring.WEIGHTS[key]} here and {raw} in the runtime"
            )

    raw_scale = _php_define(source, SCALE_SYMBOL)
    if raw_scale is None:
        found.append(f"the runtime declares no {SCALE_SYMBOL}")
    elif abs(float(raw_scale) - scoring.SCORE_SCALE) > 1e-9:
        found.append(f"the score scale is {scoring.SCORE_SCALE} here and {raw_scale} in the runtime")

    return found


def _column_findings(root: Path) -> list[str]:
    """The columns written here against the header of a recorded run."""
    recorded = root / "results" / "agent" / "baseline" / "tasks.csv"
    if not recorded.is_file():
        return []
    header = recorded.read_text(encoding="utf-8").splitlines()[0].split(",")
    missing = [name for name in runner.COLUMNS if name not in header]
    if missing:
        return [
            "this package writes column(s) a recorded run does not carry: " + ", ".join(missing)
        ]
    return []


def report(root: Path | None = None) -> int:
    """Print the check, and return the count of disagreements."""
    resolved = root or repo_root()
    if resolved is None:
        print("parity: the runtime is not on this machine, so nothing was compared")
        print("        this is expected on a hosted notebook and is not a pass")
        return 0

    found = findings(resolved)
    print(f"parity: read {resolved}")
    if not found:
        print(
            "parity: this package agrees with the runtime on the task set, "
            "the weights, the scale and the columns"
        )
        return 0
    for item in found:
        print(f"[error] parity: {item}")
    return len(found)


if __name__ == "__main__":
    raise SystemExit(1 if report() else 0)
'''

# gembench/charts.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['charts'] = r'''
"""The runs as pictures, so a reader sees the shape before reading the table.

Nine rows and five columns is read one cell at a time; the same numbers as bars
are read at once, and the reading a benchmark exists for is which task a solver
fails and whether the failure is the format, the plan, or the ending state.

The charts are written as SVG text here rather than by a plotting library. Three
reasons, in the order they matter: the notebook gains no dependency it did not
have, the file that leaves the run is a vector document that opens in a browser
and in a pull request and needs no runtime, and the geometry is stated in the
source, so every rectangle is drawn with ``rx`` and ``ry`` at zero. A bar is a
bar and not a pill: :data:`BORDER_RADIUS` is zero and every rectangle passes it.

Nothing here decides anything. A chart is a reading of the rows a run wrote, and
the rows stay the record.
"""

from __future__ import annotations

from pathlib import Path

#: Every rectangle is drawn with this radius, and it is zero on purpose: a
#: rounded bar reads as a decoration rather than a measurement, and a rounded
#: cell in the matrix reads as a status chip.
BORDER_RADIUS = 0

WIDTH = 940
LABEL_WIDTH = 250
ROW_GAP = 8
SUB_HEIGHT = 13
SUB_GAP = 2
TOP = 78
BOTTOM = 26
FOOTER_LINE = 16

AXIS_TICKS = (0, 25, 50, 75, 100)
MAXIMUM = 100.0

NUMBER_COLUMN = 52

BACKGROUND = "#ffffff"
BAR_FILL = "#2f5d8a"
BAR_EDGE = "#1d3b57"
GRID = "#d8d8d8"
AXIS = "#8a8a8a"
TEXT = "#202020"
MUTED = "#6a6a6a"
PASS_FILL = "#1f7a3d"
FAIL_FILL = "#b3261e"
MISSING_FILL = "#e4e4e4"

#: The colours the series cycle through. Stated rather than generated, so a
#: chart rebuilt from the same rows is the same picture.
SERIES_FILLS = (
    "#2f5d8a",
    "#8a6a2f",
    "#4a7a5a",
    "#7a3f6a",
    "#556070",
    "#8a4a3f",
    "#3f6f7a",
    "#6f6f3f",
)


def escape(text: object) -> str:
    """Return *text* with the three characters that end an SVG text node replaced."""
    return (
        str(text)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )


def _text(x: float, y: float, body: object, size: int = 11, anchor: str = "start", fill: str = TEXT, weight: str = "normal") -> str:
    return (
        f'<text x="{x:.1f}" y="{y:.1f}" font-family="Helvetica, Arial, sans-serif" '
        f'font-size="{size}" fill="{fill}" text-anchor="{anchor}" font-weight="{weight}">'
        f"{escape(body)}</text>"
    )


def _rect(x: float, y: float, width: float, height: float, fill: str, edge: str = "") -> str:
    stroke = f' stroke="{edge}" stroke-width="0.6"' if edge else ""
    return (
        f'<rect x="{x:.1f}" y="{y:.1f}" width="{width:.1f}" height="{height:.1f}" '
        f'rx="{BORDER_RADIUS}" ry="{BORDER_RADIUS}" fill="{fill}"{stroke}/>'
    )


def _line(x1: float, y1: float, x2: float, y2: float, stroke: str = GRID) -> str:
    return f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{stroke}" stroke-width="0.8"/>'


def _document(body: list[str], width: float, height: float, label: str = "") -> str:
    """Wrap drawn elements in one SVG document.

    *label* becomes the document's title element, which is what a reader without
    the image in front of them is given by an assistive tool.
    """
    header = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width:.0f}" height="{height:.0f}" '
        f'viewBox="0 0 {width:.0f} {height:.0f}" role="img" shape-rendering="crispEdges">'
    )
    title = f"<title>{escape(label)}</title>" if label else ""
    background = _rect(0, 0, width, height, BACKGROUND)
    return "\n".join([header, title, background] + body + ["</svg>", ""])


def _truncate(text: object, limit: int) -> str:
    body = str(text)
    return body if len(body) <= limit else body[: limit - 1] + "\u2026"


def _legend(series: list[str], x: float, y: float) -> list[str]:
    """A swatch and a name per series, drawn in one row that wraps at the width."""
    parts: list[str] = []
    cursor = x
    for index, name in enumerate(series):
        fill = SERIES_FILLS[index % len(SERIES_FILLS)]
        parts.append(_rect(cursor, y - 9, 10, 10, fill, BAR_EDGE))
        parts.append(_text(cursor + 15, y, _truncate(name, 34), size=11, fill=MUTED))
        cursor += 15 + 6.4 * min(len(name), 34) + 18
    return parts


def _axis(first_x: float, last_x: float, top: float, height: float) -> list[str]:
    parts: list[str] = []
    for tick in AXIS_TICKS:
        x = first_x + (last_x - first_x) * (tick / MAXIMUM)
        parts.append(_line(x, top, x, top + height, GRID))
        parts.append(_text(x, top - 6, tick, size=10, anchor="middle", fill=MUTED))
    return parts


def bar_chart(title: str, groups: list[tuple[str, dict[str, float]]], series: list[str], note: str = "") -> str:
    """A horizontal grouped bar chart: one row per *group*, one bar per series.

    *groups* is a list of ``(label, {series name: value})``, the value read out of
    a hundred, and a series a group does not carry is left out rather than drawn
    as zero, because a task a run did not measure and a task it scored nothing
    on are different facts.
    """
    if not groups:
        return _document(
            [_text(16, 34, "no rows to draw", size=12, fill=MUTED)],
            WIDTH,
            TOP,
            title,
        )

    per_row = len(series) * SUB_HEIGHT + (len(series) - 1) * SUB_GAP
    plot_height = len(groups) * per_row + (len(groups) - 1) * ROW_GAP
    height = TOP + plot_height + BOTTOM

    first_x = LABEL_WIDTH
    last_x = WIDTH - 74

    body: list[str] = [
        _text(16, 24, title, size=15, weight="bold"),
    ]
    if note:
        body.append(_text(16, 42, note, size=11, fill=MUTED))
    body += _legend(series, 16, 60)
    body += _axis(first_x, last_x, TOP, plot_height)

    for index, (label, values) in enumerate(groups):
        row_top = TOP + index * (per_row + ROW_GAP)
        body.append(_text(16, row_top + per_row / 2 + 4, _truncate(label, 38), size=11))
        for position, name in enumerate(series):
            if name not in values:
                continue
            value = max(0.0, min(MAXIMUM, float(values[name])))
            fill = SERIES_FILLS[position % len(SERIES_FILLS)]
            bar_x = row_top + position * (SUB_HEIGHT + SUB_GAP)
            length = (last_x - first_x) * (value / MAXIMUM)
            body.append(_rect(first_x, bar_x, max(1.0, length), SUB_HEIGHT, fill, BAR_EDGE))
            body.append(_text(last_x + 6, bar_x + SUB_HEIGHT - 2, f"{value:.1f}", size=10, fill=MUTED))

    return _document(body, WIDTH, height, title)


def matrix_chart(title: str, tasks: list[dict], labels: list[str], matrix: dict[str, dict[str, dict]], note: str = "") -> str:
    """One square per task and column: reached the end state, did not, or not measured.

    The columns are numbered rather than headed, because nine column names will
    not fit over nine columns of squares at a readable size, and the numbers are
    read off the legend underneath.
    """
    cell = 26
    gap = 4
    first_x = LABEL_WIDTH
    grid_width = len(labels) * (cell + gap)
    height = TOP + len(tasks) * (cell + gap) + BOTTOM + len(labels) * FOOTER_LINE

    body: list[str] = [_text(16, 24, title, size=15, weight="bold")]
    if note:
        body.append(_text(16, 42, note, size=11, fill=MUTED))

    for index, label in enumerate(labels):
        x = first_x + index * (cell + gap)
        body.append(_text(x + cell / 2, TOP - 10, index + 1, size=11, anchor="middle", fill=MUTED))
        body.append(_rect(x, TOP, cell, 1, AXIS))

    for row, task in enumerate(tasks):
        y = TOP + row * (cell + gap)
        body.append(_text(16, y + cell / 2 + 4, _truncate(f"{task['id']} ({task['capability']})", 44), size=11))
        for column, label in enumerate(labels):
            x = first_x + column * (cell + gap)
            cell_value = matrix.get(label, {}).get(task["id"])
            if cell_value is None:
                fill, edge, mark = MISSING_FILL, "", "?"
            elif cell_value.get("success"):
                fill, edge, mark = PASS_FILL, "", "P"
            else:
                fill, edge, mark = FAIL_FILL, "", "F"
            body.append(_rect(x, y, cell, cell, fill, edge))
            body.append(_text(x + cell / 2, y + cell / 2 + 4, mark, size=11, anchor="middle", fill="#ffffff"))

    legend_y = TOP + len(tasks) * (cell + gap) + 14
    for index, label in enumerate(labels):
        body.append(_text(16, legend_y + index * FOOTER_LINE, f"{index + 1}. {label}", size=11, fill=MUTED))
    body.append(_text(LABEL_WIDTH, legend_y + len(labels) * FOOTER_LINE - FOOTER_LINE, "P reached the end state, F did not, ? not measured", size=10, fill=MUTED))

    return _document(body, WIDTH, height, title)


def dimension_keys() -> tuple[str, ...]:
    from gembench import scoring

    return scoring.DIMENSIONS


def composite_chart(runs: list[dict]) -> str:
    """A run's per-task composite, one group per task, one bar per run."""
    series = [f"{run['side']}/{run['model']}" for run in runs]
    by_task: dict[str, dict[str, float]] = {}
    order: list[str] = []
    for run in runs:
        label = f"{run['side']}/{run['model']}"
        for row in run["rows"]:
            by_task.setdefault(row["task_id"], {})[label] = float(row["composite"])
            if row["task_id"] not in order:
                order.append(row["task_id"])
    groups = [(task_id, by_task[task_id]) for task_id in order]
    return bar_chart(
        "Composite by task, out of 100",
        groups,
        series,
        note="one bar per run and task; the composite is the weighted blend of the five dimensions",
    )


def dimension_chart(runs: list[dict], dimensions: tuple[str, ...] | None = None) -> str:
    """Each of the five dimensions per run, as its mean over the suite, out of a hundred.

    One row per dimension rather than one row per task, because the question this
    chart answers is which part of the blend a solver loses on, and that question
    is about the dimension and not about the task.
    """
    names = dimensions or dimension_keys()
    series = [f"{run['side']}/{run['model']}" for run in runs]
    by_dimension: dict[str, dict[str, float]] = {name: {} for name in names}
    for run in runs:
        label = f"{run['side']}/{run['model']}"
        for row in run["rows"]:
            for name in names:
                by_dimension[name].setdefault(label, 0.0)
                by_dimension[name][label] += float(row.get(name, 0.0))
        for name in names:
            count = max(1, len(run["rows"]))
            by_dimension[name][label] = 100.0 * by_dimension[name][label] / count

    groups = [(name, by_dimension[name]) for name in names]
    return bar_chart(
        "The five dimensions, each out of 100",
        groups,
        series,
        note="the mean over the suite of every dimension the run scored; the composite is their weighted blend",
    )


def capability_chart(runs: list[dict]) -> str:
    """A run's mean composite per capability."""
    series = [f"{run['side']}/{run['model']}" for run in runs]
    totals: dict[str, dict[str, list[float]]] = {}
    order: list[str] = []
    for run in runs:
        label = f"{run['side']}/{run['model']}"
        for row in run["rows"]:
            capability = row["capability"]
            bucket = totals.setdefault(capability, {})
            bucket.setdefault(label, []).append(float(row["composite"]))
            if capability not in order:
                order.append(capability)
    groups = [
        (capability, {label: sum(values) / len(values) for label, values in totals[capability].items()})
        for capability in order
    ]
    return bar_chart(
        "Composite by capability, out of 100",
        groups,
        series,
        note="the same tasks grouped by what they measure, so a weakness names an ability",
    )


def summary_chart(runs: list[dict]) -> str:
    """A run in one row of bars: the composite, the pass rate, and the share of the budget spent."""
    from gembench import scoring

    series = [f"{run['side']}/{run['model']}" for run in runs]
    by_model: dict[str, float] = {}
    passed: dict[str, float] = {}
    spent: dict[str, float] = {}
    for run in runs:
        label = f"{run['side']}/{run['model']}"
        rows = run["rows"]
        by_model[label] = sum(float(row["composite"]) for row in rows) / len(rows) if rows else 0.0
        passed[label] = 100.0 * (sum(1 for row in rows if row["success"]) / len(rows) if rows else 0.0)
        spent[label] = 100.0 * (
            sum(float(row["steps_used"]) / max(1.0, float(row["budget"])) for row in rows) / len(rows)
            if rows
            else 0.0
        )
    groups = [("composite", by_model), ("tasks passed", passed), ("budget spent", spent)]
    return bar_chart(
        "A run in one line",
        groups,
        series,
        note=f"the composite is the blend of the five dimensions, reported out of {scoring.SCORE_SCALE:.0f}",
    )


def write(out_dir: Path, runs: list[dict]) -> dict[str, str]:
    """Write every chart a run supports, and return the path of each."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    named = {
        "summary": summary_chart(runs),
        "composite": composite_chart(runs),
        "dimensions": dimension_chart(runs),
        "capability": capability_chart(runs),
    }
    written: dict[str, str] = {}
    for name, body in named.items():
        path = out_dir / f"{name}.svg"
        path.write_text(body, encoding="utf-8")
        written[name] = str(path)
    return written


def matrix(out_dir: Path, document: dict) -> str:
    """Write the pass matrix of a comparison, which is the chart that needs both sides."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    labels = list(document["columns"])
    body = matrix_chart(
        "End state reached, by task and by column",
        document["tasks"],
        labels,
        document["matrix"],
        note="a column is one side and one solver label; a row is one task",
    )
    path = out_dir / "matrix.svg"
    path.write_text(body, encoding="utf-8")
    return str(path)


def figure(paths: dict[str, str]) -> str:
    """The charts as one HTML fragment, for a page that holds the files beside it."""
    parts: list[str] = []
    for name, path in sorted(paths.items()):
        target = Path(path)
        if not target.is_file():
            continue
        parts.append(f'<h3>{escape(name)}</h3>')
        parts.append(target.read_text(encoding="utf-8"))
    return "\n".join(parts) + "\n"


def show(paths: dict[str, str]) -> list[str]:
    """Display each chart in a notebook, and report the ones that could not be shown.

    In a notebook the SVG is displayed inline. Anywhere else the paths are
    returned, because a chart a reader cannot see would otherwise be a silent
    omission rather than a file on disk.
    """
    shown: list[str] = []
    try:
        from IPython.display import SVG, display  # type: ignore
    except ImportError:
        return [str(path) for path in paths.values()]

    for name, path in sorted(paths.items()):
        target = Path(path)
        if not target.is_file():
            continue
        display(SVG(filename=str(target)))
        shown.append(name)
    return shown
'''

# gembench/dataset.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['dataset'] = r'''
"""The runs as one dataset, and the reading that turns it into work on the harness.

A benchmark result is only useful if something can be done with it. The tables
this module writes are for that: every run flattened to one row per side, solver
and task, with the counters the loop recorded left in their own columns rather
than folded into the composite, so a row can be filtered and joined by something
other than this file.

On top of the rows sits the part that names the work. Every failed row is read
against a small table of signals, each of which is a measurable fact about the
row rather than an opinion about it, and each of which names the part of the
harness it points at: the output contract, the tool catalog, the argument
schema, the retry rule, the stop condition, the plan, or the verifier. The
signals are stated as data in :data:`SIGNALS`, the reading they support is a
report in ``learning.json``, and the mapping from signal to part of the harness
is deliberately short: it says where to look, not what to change.

The dataset is what a model-side improvement is measured against later, so the
rows carry the solver label and the side and nothing about this machine.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from gembench import compare, runner, scoring  # noqa: E402

SCHEMA_VERSION = "1"

#: The column a row carries to say which side of the run it came from. It is not
#: in :data:`runner.COLUMNS` because a run written on a machine does not know it
#: is the local side.
SIDE_COLUMN = "side"

#: The two columns this module adds after the run's own.
SIGNAL_COLUMN = "signals"
AREA_COLUMN = "areas"

#: The measurements a failed row is read against, and the part of the harness
#: each one points at. The predicate is named here and defined in
#: :data:`SIGNAL_TESTS`, so the table and the arithmetic sit next to each other.
SIGNALS: tuple[dict[str, str], ...] = (
    {
        "name": "ran_out_of_turns",
        "area": "loop_control",
        "note": "the episode spent its whole step budget: the stop condition or the length of a plan is the lever",
    },
    {
        "name": "failed_with_turns_left",
        "area": "planning",
        "note": "the episode failed while steps remained, so the plan was wrong rather than long",
    },
    {
        "name": "unstructured_action",
        "area": "output_contract",
        "note": "an intended action never became a structured call: the format the loop reads is the lever",
    },
    {
        "name": "unknown_tool",
        "area": "tool_catalog",
        "note": "a tool was named that the harness does not have: the catalog or the way it is prompted is the lever",
    },
    {
        "name": "invalid_arguments",
        "area": "argument_schema",
        "note": "a call was refused on its arguments: the schema or the examples beside it are the lever",
    },
    {
        "name": "tool_error_unrecovered",
        "area": "error_handling",
        "note": "a tool error was met and not recovered from: the retry rule is the lever",
    },
    {
        "name": "repeated_failing_turn",
        "area": "error_handling",
        "note": "the same failing turn was repeated: the loop needs to change what it does on a repeat",
    },
    {
        "name": "redundant_work",
        "area": "efficiency",
        "note": "calls were repeated that a shorter path skips: the plan or the memory of what was already done is the lever",
    },
    {
        "name": "partial_checks",
        "area": "verification",
        "note": "some of the task's own checks passed: the verifier names exactly what the end state missed",
    },
    {
        "name": "answer_not_given",
        "area": "stopping_criteria",
        "note": "the honest answer was not given on a task whose goal is to stop: refusal has to be a teachable output",
    },
)


def _number(row: dict, key: str) -> float:
    try:
        return float(row.get(key, 0) or 0)
    except (TypeError, ValueError):
        return 0.0


def _flag(row: dict, key: str) -> bool:
    return _number(row, key) > 0.5


def _ran_out_of_turns(row: dict) -> bool:
    return _flag(row, "budget_exhausted") and not _flag(row, "success")


def _failed_with_turns_left(row: dict) -> bool:
    return not _flag(row, "success") and not _flag(row, "budget_exhausted")


def _unstructured_action(row: dict) -> bool:
    return _number(row, "unstructured_tool_calls") > 0


def _unknown_tool(row: dict) -> bool:
    return _number(row, "unknown_tools") > 0


def _invalid_arguments(row: dict) -> bool:
    return _number(row, "invalid_args") > 0


def _tool_error_unrecovered(row: dict) -> bool:
    return _number(row, "tool_errors") > 0 and not _flag(row, "recovered")


def _repeated_failing_turn(row: dict) -> bool:
    return _number(row, "repeated_failed_turns") > 0


def _redundant_work(row: dict) -> bool:
    return _number(row, "redundant_calls") > 0


def _partial_checks(row: dict) -> bool:
    total = _number(row, "checks_total")
    return total > 0 and _number(row, "checks_passed") < total


def _answer_not_given(row: dict) -> bool:
    return row.get("capability") == "stopping_criteria" and not _flag(row, "success")


#: Signal name to the reading of one row. A row that passes is read too, and
#: reads as nothing, which is why every test states the failure it looks for.
SIGNAL_TESTS = {
    "ran_out_of_turns": _ran_out_of_turns,
    "failed_with_turns_left": _failed_with_turns_left,
    "unstructured_action": _unstructured_action,
    "unknown_tool": _unknown_tool,
    "invalid_arguments": _invalid_arguments,
    "tool_error_unrecovered": _tool_error_unrecovered,
    "repeated_failing_turn": _repeated_failing_turn,
    "redundant_work": _redundant_work,
    "partial_checks": _partial_checks,
    "answer_not_given": _answer_not_given,
}


def read_run_rows(directory: Path) -> dict[str, dict]:
    """One run's rows, every column kept as it was written."""
    path = Path(directory) / "tasks.csv"
    with path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"{path} is empty")
        missing = [name for name in compare.REQUIRED_COLUMNS if name not in reader.fieldnames]
        if missing:
            raise ValueError(f"{path} is missing column(s): {', '.join(missing)}")
        rows: dict[str, dict] = {}
        for raw in reader:
            task_id = str(raw.get("task_id", ""))
            if task_id:
                rows[task_id] = dict(raw)
        return rows


def read_sides(sides: dict[str, Path]) -> list[dict]:
    """Every run under every side, with the side and the run's directory attached."""
    runs: list[dict] = []
    for side, root in sides.items():
        for directory in compare.run_directories(Path(root)):
            try:
                rows = read_run_rows(directory)
            except (ValueError, OSError):
                continue
            if not rows:
                continue
            first = next(iter(rows.values()))
            manifest = {}
            manifest_path = directory / "manifest.json"
            if manifest_path.is_file():
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            runs.append(
                {
                    "side": side,
                    "model": str(first.get("model", directory.name)),
                    "run_id": str(first.get("run_id", directory.name)),
                    "suite": str(manifest.get("run", {}).get("suite", "")),
                    "profile": str(first.get("tool_mode", "")),
                    "directory": str(directory),
                    "rows": list(rows.values()),
                }
            )
    return runs


def signals_for(row: dict) -> list[str]:
    return [name for name, test in SIGNAL_TESTS.items() if test(row)]


def area_of(signal: str) -> str:
    for entry in SIGNALS:
        if entry["name"] == signal:
            return entry["area"]
    return ""


def areas_for(row: dict) -> list[str]:
    seen: list[str] = []
    for signal in signals_for(row):
        area = area_of(signal)
        if area and area not in seen:
            seen.append(area)
    return seen


def flatten(runs: list[dict]) -> list[dict]:
    """One row per side, solver and task, in the run's own column order."""
    flat: list[dict] = []
    for run in runs:
        for row in run["rows"]:
            record = {SIDE_COLUMN: run["side"]}
            for column in runner.COLUMNS:
                record[column] = row.get(column, "")
            found = signals_for(row)
            record[SIGNAL_COLUMN] = " ".join(found)
            record[AREA_COLUMN] = " ".join(areas_for(row))
            flat.append(record)
    return flat


def columns() -> list[str]:
    return [SIDE_COLUMN] + list(runner.COLUMNS) + [SIGNAL_COLUMN, AREA_COLUMN]


def records(runs: list[dict]) -> list[dict]:
    """The same rows as objects, with the counters and the checks kept separate.

    A flat CSV is what a spreadsheet and a join want; a record that keeps the
    counters in their own object is what a training script wants, because a
    counter is not a feature until someone says it is.
    """
    out: list[dict] = []
    for run in runs:
        for row in run["rows"]:
            found = signals_for(row)
            out.append(
                {
                    "schema_version": SCHEMA_VERSION,
                    "side": run["side"],
                    "solver": run["model"],
                    "run_id": run["run_id"],
                    "tool_mode": str(row.get("tool_mode", "")),
                    "task": {
                        "id": str(row.get("task_id", "")),
                        "capability": str(row.get("capability", "")),
                        "budget": int(_number(row, "budget")),
                    },
                    "outcome": {
                        "success": _flag(row, "success"),
                        "composite": _number(row, "composite"),
                        "checks_passed": int(_number(row, "checks_passed")),
                        "checks_total": int(_number(row, "checks_total")),
                    },
                    "dimensions": {name: _number(row, name) for name in scoring.DIMENSIONS},
                    "counters": {name: int(_number(row, name)) for name in scoring.COUNTERS},
                    "cost": {
                        "steps_used": int(_number(row, "steps_used")),
                        "budget_exhausted": _flag(row, "budget_exhausted"),
                        "latency_ms": _number(row, "latency_ms"),
                    },
                    "signals": found,
                    "areas": areas_for(row),
                }
            )
    return out


def learning(runs: list[dict]) -> dict:
    """Where the lost composite sits, grouped by the part of the harness that a signal names.

    The figure that orders the work is the composite a solver did not get: a
    dimension averaged over the suite hides the four tasks a solver missed while
    scoring near a hundred on the five it passed, and the four tasks are the
    work.
    """
    by_area: dict[str, dict] = {}
    by_capability: dict[str, dict] = {}
    by_solver: dict[str, dict] = {}

    for run in runs:
        label = f"{run['side']}/{run['model']}"
        solver = by_solver.setdefault(
            label, {"tasks": 0, "passed": 0, "lost": 0.0, "dimensions": {name: 0.0 for name in scoring.DIMENSIONS}}
        )
        for row in run["rows"]:
            solver["tasks"] += 1
            solver["passed"] += 1 if _flag(row, "success") else 0
            solver["lost"] = round(solver["lost"] + (scoring.SCORE_SCALE - _number(row, "composite")), 4)
            for name in scoring.DIMENSIONS:
                solver["dimensions"][name] = round(solver["dimensions"][name] + _number(row, name), 4)

            capability = str(row.get("capability", ""))
            bucket = by_capability.setdefault(
                capability, {"tasks": 0, "passed": 0, "lost": 0.0, "solvers": []}
            )
            bucket["tasks"] += 1
            bucket["passed"] += 1 if _flag(row, "success") else 0
            bucket["lost"] = round(bucket["lost"] + (scoring.SCORE_SCALE - _number(row, "composite")), 4)
            if label not in bucket["solvers"]:
                bucket["solvers"].append(label)

            for signal in signals_for(row):
                area = area_of(signal)
                if not area:
                    continue
                group = by_area.setdefault(
                    area, {"tasks": 0, "lost": 0.0, "signal_counts": {}, "tasks_named": [], "solvers": [], "note": ""}
                )
                group["tasks"] += 1
                group["lost"] = round(group["lost"] + (scoring.SCORE_SCALE - _number(row, "composite")), 4)
                group["signal_counts"][signal] = group["signal_counts"].get(signal, 0) + 1
                task_id = str(row.get("task_id", ""))
                if task_id not in group["tasks_named"]:
                    group["tasks_named"].append(task_id)
                if label not in group["solvers"]:
                    group["solvers"].append(label)
                group["note"] = next((entry["note"] for entry in SIGNALS if entry["name"] == signal), "")

    for label, solver in by_solver.items():
        solver["dimensions"] = {
            name: round(value / max(1, solver["tasks"]), 4) for name, value in solver["dimensions"].items()
        }

    ordered = sorted(by_area.items(), key=lambda item: (-item[1]["lost"], item[0]))
    return {
        "document": "benchmark-learning",
        "schema_version": SCHEMA_VERSION,
        "solvers": by_solver,
        "capabilities": by_capability,
        "areas": {name: values for name, values in ordered},
        "priority": [name for name, _ in ordered],
    }


def render(reading: dict) -> str:
    """The reading as lines, with the work ordered by the composite it accounts for."""
    lines = ["Where the lost composite sits", ""]
    lines.append("  one row per side, solver and task, so a solver measured on two sides is counted twice")
    lines.append("")
    if not reading["areas"]:
        lines.append("  no failed row carried a signal, so nothing is named")
        return "\n".join(lines) + "\n"

    lines.append(f"  {'area':<20} {'tasks':>5} {'lost':>8}  signals")
    for area, values in reading["areas"].items():
        counts = ", ".join(f"{name} {count}" for name, count in sorted(values["signal_counts"].items()))
        lines.append(f"  {area:<20} {values['tasks']:>5} {values['lost']:>8.2f}  {counts}")

    lines += ["", "  the lever beside each area", ""]
    for area, values in reading["areas"].items():
        lines.append(f"    {area}: {values['note']}")
    lines += ["", "  by capability", ""]
    for capability, values in sorted(reading["capabilities"].items()):
        lines.append(
            f"    {capability:<24} passed {values['passed']}/{values['tasks']}  lost {values['lost']:.2f}"
        )
    lines += ["", "  by solver", ""]
    for label, values in sorted(reading["solvers"].items()):
        lines.append(f"    {label:<34} passed {values['passed']}/{values['tasks']}  lost {values['lost']:.2f}")
    return "\n".join(lines) + "\n"


def write(out_dir: Path, runs: list[dict], document: dict | None = None) -> dict[str, str]:
    """Write the dataset, the records, the signal table and the reading."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    flat = flatten(runs)
    csv_path = out_dir / "dataset.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns())
        writer.writeheader()
        for row in flat:
            writer.writerow({name: row.get(name, "") for name in columns()})

    rows = records(runs)
    jsonl_path = out_dir / "dataset.jsonl"
    with jsonl_path.open("w", encoding="utf-8") as handle:
        for entry in rows:
            handle.write(json.dumps(entry, sort_keys=False) + "\n")

    signals_path = out_dir / "signals.json"
    signals_path.write_text(
        json.dumps(
            {
                "document": "benchmark-signals",
                "schema_version": SCHEMA_VERSION,
                "signals": [dict(entry) for entry in SIGNALS],
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    reading = learning(runs)
    learning_path = out_dir / "learning.json"
    learning_path.write_text(json.dumps(reading, indent=2) + "\n", encoding="utf-8")

    note_path = out_dir / "learning.txt"
    note_path.write_text(render(reading), encoding="utf-8")

    descriptor = {
        "document": "benchmark-dataset",
        "schema_version": SCHEMA_VERSION,
        "rows": len(flat),
        "records": len(rows),
        "columns": columns(),
        "sides": sorted({run["side"] for run in runs}),
        "solvers": sorted({f"{run['side']}/{run['model']}" for run in runs}),
        "areas": reading["priority"],
        "files": {
            "csv": csv_path.name,
            "jsonl": jsonl_path.name,
            "signals": signals_path.name,
            "learning": learning_path.name,
            "reading": note_path.name,
        },
    }
    if document is not None:
        descriptor["comparison"] = {
            "columns": list(document.get("columns", {})),
            "tasks": len(document.get("tasks", [])),
            "findings": list(document.get("findings", [])),
        }
    descriptor_path = out_dir / "dataset.json"
    descriptor_path.write_text(json.dumps(descriptor, indent=2) + "\n", encoding="utf-8")

    return {
        "csv": str(csv_path),
        "jsonl": str(jsonl_path),
        "signals": str(signals_path),
        "learning": str(learning_path),
        "reading": str(note_path),
        "descriptor": str(descriptor_path),
    }


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="gembench.dataset",
        description="Write the runs as one dataset and read the failures as work.",
    )
    parser.add_argument("--side", action="append", default=[], metavar="NAME=PATH")
    parser.add_argument("--out", default="", help="directory to write the dataset into")
    parser.add_argument("--json", action="store_true", help="print the reading instead of the table")
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    sides: dict[str, Path] = {}
    for item in args.side:
        if "=" not in item:
            print(f"--side expects NAME=PATH, got {item!r}")
            return 2
        name, _, path = item.partition("=")
        sides[name.strip()] = Path(path.strip()).expanduser()

    if not sides:
        print("at least one --side NAME=PATH is needed")
        return 2

    runs = read_sides(sides)
    if not runs:
        print("no run was read from " + ", ".join(f"{name}={root}" for name, root in sides.items()))
        return 3

    reading = learning(runs)
    if args.json:
        print(json.dumps(reading, indent=2))
    else:
        print(render(reading), end="")

    if args.out:
        files = write(Path(args.out), runs)
        print()
        for name, path in files.items():
            print(f"wrote {path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

# gembench/bundle.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['bundle'] = r'''
"""Everything one benchmark run produced, in one file, with a link to take it away.

A hosted run ends when its session does and the files in it are gone. A run whose
result cannot be carried off the machine it ran on is not evidence, so the last
step of the notebook writes every file it produced into one zip and prints a
link to it: the runs, the comparison, the charts, the dataset, and a manifest
that names each member with its size and a digest, so a reader can tell what
they hold without unpacking it.

The link is produced by the notebook's own machinery when there is any, and the
path is printed when there is not, because a link that silently was not rendered
would read as a run that produced nothing. Nothing here reaches the network: the
zip is written beside the run and the link points at it.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

SCHEMA_VERSION = "1"

DEFAULT_NAME = "benchmark-bundle.zip"

#: The manifest's own name. It is written fresh on every run and is carried in
#: the zip, but it is not one of the files it describes: a digest of the file
#: that holds the digests is a circle, so the manifest names every member but
#: itself and says so.
MANIFEST_NAME = "bundle.json"

#: Directories under the run that are not the run: a build byproduct and a
#: cache are never part of what a reader is being given.
SKIP_NAMES = {".DS_Store", "__pycache__"}


def digest(path: Path) -> str:
    """A short content digest, which is what tells two runs' charts apart."""
    body = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(65536), b""):
            body.update(block)
    return body.hexdigest()[:16]


def collect(out_dir: Path, target: Path) -> list[Path]:
    """Every file under *out_dir* that belongs in the bundle, the target zip and the manifest aside."""
    out_dir = Path(out_dir)
    target = Path(target)
    found: list[Path] = []
    for item in sorted(out_dir.rglob("*")):
        if not item.is_file():
            continue
        if item == target or item.suffix == ".zip":
            continue
        if item.name == MANIFEST_NAME:
            continue
        if any(part in SKIP_NAMES for part in item.parts):
            continue
        found.append(item)
    return found


def manifest(out_dir: Path, members: list[Path]) -> dict:
    """What the bundle holds, by name, by size and by digest."""
    out_dir = Path(out_dir)
    entries = []
    for member in members:
        entries.append(
            {
                "path": member.relative_to(out_dir).as_posix(),
                "bytes": member.stat().st_size,
                "sha256": digest(member),
            }
        )
    kinds: dict[str, int] = {}
    for entry in entries:
        top = entry["path"].split("/")[0]
        kinds[top] = kinds.get(top, 0) + 1
    return {
        "document": "benchmark-bundle",
        "schema_version": SCHEMA_VERSION,
        "created": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "manifest": MANIFEST_NAME,
        "members": entries,
        "totals": {
            "members": len(entries),
            "bytes": sum(entry["bytes"] for entry in entries),
        },
        "containing": {name: count for name, count in sorted(kinds.items())},
    }


def build(out_dir: Path, target: Path | None = None) -> dict:
    """Write one zip of everything under *out_dir*, and return what it holds.

    The manifest is written into the directory before the zip is made and is
    carried in the zip as a member, so the same account travels with the files
    and is readable without unpacking them. Its own size and digest are not in
    it, for the reason stated on :data:`MANIFEST_NAME`.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    target = Path(target) if target is not None else out_dir / DEFAULT_NAME

    members = collect(out_dir, target)
    body = manifest(out_dir, members)
    manifest_path = out_dir / MANIFEST_NAME
    manifest_path.write_text(json.dumps(body, indent=2) + "\n", encoding="utf-8")

    target.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        for member in members:
            archive.write(member, member.relative_to(out_dir).as_posix())
        archive.write(manifest_path, MANIFEST_NAME)

    return {
        "path": str(target),
        "size": target.stat().st_size,
        "members": [entry["path"] for entry in body["members"]] + [MANIFEST_NAME],
        "counts": body["totals"],
        "containing": body["containing"],
        "manifest": str(manifest_path),
    }


def link(path: Path, label: str = "") -> str:
    """The HTML a notebook renders to offer the file, on a Kaggle page or anywhere else."""
    target = Path(path)
    name = label or target.name
    return (
        f'<a href="{target.as_uri()}" download="{target.name}" '
        f'style="border-radius:0;text-decoration:none;'
        f'font-family:Helvetica,Arial,sans-serif;font-size:14px;'
        f'padding:8px 12px;display:inline-block;background:#2f5d8a;color:#ffffff">'
        f"download {name}</a>"
    )


def show(path: Path, label: str = "") -> str:
    """Render a download link where there is a notebook, and report the file where there is not.

    The link is the notebook's own ``FileLink`` and not an anchor this module
    writes, because on a hosted page the file is served through the notebook's
    own route and a ``file://`` href would point at a path the reader's browser
    cannot reach. :func:`link` is kept for a page that holds the file beside it.
    """
    target = Path(path)
    size = target.stat().st_size if target.is_file() else 0
    text = f"{target} ({size / 1024:.1f} KiB)"
    try:
        from IPython.display import FileLink, display  # type: ignore
    except ImportError:
        return text
    display(FileLink(str(target), result_html_prefix=f"{label or target.name}: "))
    return text


def render(built: dict) -> str:
    lines = [
        f"Benchmark bundle: {built['path']}",
        f"  members   {built['counts']['members']}",
        f"  bytes     {built['counts']['bytes']}",
        "",
    ]
    for name, count in built["containing"].items():
        lines.append(f"    {name:<16} {count} file(s)")
    return "\n".join(lines) + "\n"


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="gembench.bundle",
        description="Write one zip of everything a benchmark run produced.",
    )
    parser.add_argument("--dir", required=True, help="directory holding the run's files")
    parser.add_argument("--name", default=DEFAULT_NAME, help="name of the zip inside that directory")
    parser.add_argument("--json", action="store_true", help="print the manifest instead of the table")
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    out_dir = Path(args.dir).expanduser()
    if not out_dir.is_dir():
        print(f"{out_dir} is not a directory", file=sys.stderr)
        return 3
    built = build(out_dir, out_dir / args.name)
    if args.json:
        print(json.dumps(built, indent=2))
    else:
        print(render(built), end="")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

# gembench/report.py
#
# The module is held as text and executed by the loader cell, so the code
# that runs on the machine and the code that runs here are one file.

MODULES['report'] = r'''
"""The last step of a run: the pictures, the dataset, and one file that carries both.

A benchmark that ends with a table of numbers on a screen has produced a
screenshot. This module is the step that turns a run into three things a next
person can use: the runs as pictures, so the shape of the failure is read at
once; the runs as one dataset with the failures read as work, so an improvement
can be measured against it; and one zip holding all of it with a manifest and a
link, so it leaves the machine it was measured on.

It is deliberately thin. Every figure, every row and every byte is produced by
:mod:`gembench.charts`, :mod:`gembench.dataset` and :mod:`gembench.bundle`; this
file only decides the order and the directory. The notebook's closing cells call
it, and the same call runs on a machine, so the two sides produce the same
artifacts from the same files.
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

if __package__ in (None, ""):
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from gembench import bundle, charts, compare, dataset  # noqa: E402

BUNDLE_NAME = bundle.DEFAULT_NAME


def build(sides: dict[str, Path], out_dir: Path, bundle_name: str = BUNDLE_NAME) -> dict:
    """Read the sides, write the comparison, the charts, the dataset and the zip."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    runs = dataset.read_sides(sides)
    table, problems = compare.collect(sides)
    document = compare.build(table)
    document["findings"].extend(problems)

    comparison = compare.write(out_dir, document)

    # One directory for the pictures, and it is the same one the notebook's
    # picture step writes to, so a figure is drawn once and not twice under two
    # names.
    figures_dir = out_dir / "figures"
    figures = charts.write(figures_dir, runs)
    if document["columns"]:
        figures["matrix"] = charts.matrix(figures_dir, document)

    dataset_files = dataset.write(out_dir / "dataset", runs, document)
    reading = dataset.learning(runs)

    built = bundle.build(out_dir, out_dir / bundle_name)

    return {
        "runs": runs,
        "document": document,
        "reading": reading,
        "comparison": comparison,
        "charts": figures,
        "dataset": dataset_files,
        "bundle": built,
    }


def render(record: dict) -> str:
    """The step as lines: what was read, what was written, and where the lost score sits."""
    lines = [f"read {len(record['runs'])} run(s)"]
    for run in record["runs"]:
        passed = sum(1 for row in run["rows"] if str(row.get("success", "")) in {"1", "True", "true"})
        lines.append(f"  {run['side']}/{run['model']:<20} tasks {len(run['rows'])}  passed {passed}")
    lines += ["", "wrote"]
    for name, path in sorted({**record["comparison"], **record["charts"], **record["dataset"]}.items()):
        lines.append(f"  {name:<12} {path}")

    areas = record["reading"]["areas"]
    if areas:
        lines += ["", "the work, ordered by the composite it accounts for", ""]
        for area, values in list(areas.items())[:5]:
            lines.append(f"  {area:<20} lost {values['lost']:>8.2f} over {values['tasks']} row(s)")
            lines.append(f"    {values['note']}")
    else:
        lines += ["", "no failed row carried a signal, so no area is named"]

    lines += ["", bundle.render(record["bundle"]).rstrip("\n")]
    return "\n".join(lines) + "\n"


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="gembench.report",
        description="Write the pictures, the dataset and the bundle of a benchmark run.",
    )
    parser.add_argument("--side", action="append", default=[], metavar="NAME=PATH")
    parser.add_argument("--out", default="", help="directory the run's files are under, and where the bundle is written")
    parser.add_argument("--name", default=BUNDLE_NAME, help="name of the zip inside that directory")
    parser.add_argument("--json", action="store_true", help="print the reading and the manifest as JSON")
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    sides: dict[str, Path] = {}
    for item in args.side:
        if "=" not in item:
            print(f"--side expects NAME=PATH, got {item!r}")
            return 2
        name, _, path = item.partition("=")
        sides[name.strip()] = Path(path.strip()).expanduser()

    if not sides:
        print("at least one --side NAME=PATH is needed")
        return 2
    if not args.out:
        print("--out is needed: the run's files and the bundle share one directory")
        return 2

    record = build(sides, Path(args.out), args.name)
    if not record["runs"]:
        print("no run was read from " + ", ".join(f"{name}={root}" for name, root in sides.items()))
        return 3

    if args.json:
        print(json.dumps({"reading": record["reading"], "bundle": record["bundle"]}, indent=2))
    else:
        print(render(record), end="")
    return 1 if record["document"]["findings"] else 0


if __name__ == "__main__":
    raise SystemExit(main())
'''


### The loader

Each held source is compiled and executed into its own module, registered
under `gembench`, so the modules import each other exactly as they do on a
machine. This is the step that makes the same file run in two places.


In [ ]:
import types

_package = types.ModuleType('gembench')
_package.__path__ = []
sys.modules['gembench'] = _package

for _name in MODULE_ORDER:
    _module = types.ModuleType('gembench.' + _name)
    _module.__package__ = 'gembench'
    sys.modules['gembench.' + _name] = _module
    exec(compile(MODULES[_name], 'gembench/' + _name + '.py', 'exec'), _module.__dict__)
    setattr(_package, _name, _module)

from gembench import suite, scoring, reference, runner, compare, parity
from gembench import charts, dataset, bundle, report

print('loaded   :', ', '.join('gembench.' + name for name in MODULE_ORDER))
print('isolated :', all(m in MODULE_ORDER for m in MODULES))


## 2. The suite

Nine tasks. Six measure instruction following, tool use, composition,
recovery, structured output and a conditional action. Three measure what a
solver does when the request is noisy, when the workspace is a mess, and
when the honest answer is to refuse rather than to keep trying.

Each task carries a step budget and a verifier. A task is included only if
a program can decide whether it finished.


In [ ]:
print(f"{'task':<26} {'capability':<24} {'budget':>6}  checks")
for task in suite.suite('all'):
    print(f"{task['id']:<26} {task['capability']:<24} {task['budget']:>6}  verifier attached")
print()
print('suites:', ', '.join(suite.suite_names()))
for name in suite.suite_names():
    print(f'  {name:<8} {len(suite.suite(name))} task(s)')


## 3. The arithmetic

One trajectory becomes five numbers. The composite is a blend rather than
a pass or a fail, because for a small solver the interesting result is
usually not that it failed but how it failed: a solver that plans correctly
and cannot hold the output format has a different problem, and a different
fix, from one that holds the format and never reaches the right state.

Recovery is averaged only over the tasks that actually produced an error to
recover from. Averaging it over every task would let a solver that never
needed to recover dilute a total failure to recover.


In [ ]:
weights = scoring.WEIGHTS
for name in scoring.DIMENSIONS:
    print(f'  {name:<22} {weights[name]:.2f}')
print(f'  {"sum":<22} {sum(weights.values()):.2f}')
print(f'  reported out of {scoring.SCORE_SCALE:.0f}')


## 4. The solvers

A benchmark whose only input is a language model cannot be checked: the
same suite gives a different answer twice, and a broken verifier looks
exactly like a weak solver. Two deterministic profiles separate the two.

* `reference` carries out every task correctly. It is the calibration: a
  suite a correct solver cannot pass is a broken suite, not a hard one.
* `naive` reads the request and does not check the result. It trusts a file
  name it was told might be wrong, does arithmetic in its head, emits a
  nearly-JSON object, treats tidying as deleting, and invents a value for
  an input that is not there rather than saying it is not there.

A live model is a third profile in the same shape, which is the whole
extension point.


In [ ]:
print('profiles:', ', '.join(reference.profile_names()))

rows = []
for profile in reference.profile_names():
    result = runner.run(profile, 'all', profile + '-solver', profile + '-solver')
    rows.append((profile, result))
    print(f"  {profile:<10} passed {result['summary']['tasks_passed']}/9")


## 5. The run, written down

The calibration first: if the reference profile does not pass every task,
the fault is in the suite and not in the solver, and it is reported as a
fault.


In [ ]:
findings = runner.check(rows[0][1])
if findings:
    for item in findings:
        print('[error]', item)
else:
    print('the reference profile passed every task, so the suite is answerable')


## 6. Comparing two runtimes

The comparison reads written runs, not live ones. That is what makes it
usable at all: the two sides rarely run at the same time, and a comparison
that needed both live would be a screenshot rather than a result.

Where two columns share a solver label they are expected to agree. Where
they differ, the difference is the finding.


## 7. Run the benchmark here

The same call as on a machine, writing the same three things per run: a
flat CSV, a manifest, and the printed table.


In [ ]:
from pathlib import Path

side_dir = Path(WORKING_DIR) / KAGGLE_SIDE
written = []
for profile, result in rows:
    files = runner.write_run(side_dir / result['run_id'], result,
                             extra={'side': KAGGLE_SIDE, 'runtime': platform.platform()})
    written.append(files)
    print(f"{profile:<10} -> {files['tasks']}")
print()
print(runner.render(rows[0][1]), end='')


## 8. The hosted side, on its own

Before the two sides are joined, the hosted side is read back from the
files it just wrote. A run that cannot be read back is not evidence.


In [ ]:
table, problems = compare.collect({KAGGLE_SIDE: side_dir})
document = compare.build(table)
document['findings'].extend(problems)
print(compare.render({'columns': document['columns'], 'tasks': document['tasks'],
                     'matrix': document['matrix'], 'sides': document['sides'],
                     'findings': document['findings'], 'disagreements': document['disagreements']}), end='')


## 9. The two sides beside each other

The local side is read from a results directory published as a dataset and
attached to this notebook at `/kaggle/input/agent-benchmark-local`. It holds the same flat
CSV the run just wrote, produced by the same code on a different machine.

If the directory is not attached, the step says so and prints the command
that produces it, rather than reporting a comparison it could not make. The
table is printed here; it is written, with everything else, in the last step.


In [ ]:
from pathlib import Path

# The sides this run knows about. The hosted side is always there; the local
# side is added only when the directory is attached, so the list is the
# description of what was actually measured.
sides = {KAGGLE_SIDE: side_dir}

local_dir = Path(LOCAL_RESULTS_DIR)
if compare.run_directories(local_dir):
    sides[LOCAL_SIDE] = local_dir
    both, more = compare.collect(sides)
    joined = compare.build(both)
    joined['findings'].extend(more)
    print(compare.render(joined), end='')
else:
    print('no local results at', local_dir)
    print()
    print('To make them, run the same benchmark on the machine that holds the models:')
    print()
    print('  python3 tools/kaggle/gembench/runner.py --profile reference --suite all \\')
    print('      --out results/benchmark/local')
    print('  python3 tools/kaggle/gembench/runner.py --profile naive --suite all \\')
    print('      --out results/benchmark/local')
    print()
    print('then publish results/benchmark/local as a dataset and attach it here.')


## 10. The same reading on a machine

Everything this notebook writes is read by the same code that wrote it, on
any machine, with no rerun. On the machine that ran the local side, the
hosted run is pulled into a directory and the whole reading is rebuilt from
the files:

```bash
python3 tools/kaggle/gembench/compare.py \
    --side local=results/benchmark/local \
    --side kaggle=results/benchmark/kaggle \
    --out results/benchmark
```

The exit code of the comparison is non-zero when the two sides did not
measure the same task set, which is a finding about the runs rather than
about the solvers. One call writes the whole set this notebook ends with:

```bash
python3 tools/kaggle/gembench/report.py --side local=results/benchmark/local --side kaggle=results/benchmark/kaggle --out results/benchmark
```

It writes the comparison, the pictures, the dataset and the zip.


## 11. The runs as pictures

A table of nine rows is read one cell at a time. The same numbers as bars are
read at once, and the reading a benchmark exists for is which task a solver
fails and whether the failure is the format, the plan, or the ending state.

The charts are SVG written by the benchmark itself rather than by a plotting
library. Two consequences: the notebook gains no dependency, and the file
that leaves it is a vector document that opens in a browser and in a pull
request. Every rectangle is drawn with a zero corner radius, which is stated
in the source as a constant rather than left to a style sheet, because a
rounded bar reads as a decoration rather than a measurement.

The matrix is the chart that needs both sides: one column per side and solver
label, one row per task, and a cell that says whether the end state was
reached rather than how near it was.


In [ ]:
here = Path(WORKING_DIR) / KAGGLE_SIDE
runs_here = dataset.read_sides({KAGGLE_SIDE: here})

figures = charts.write(Path(WORKING_DIR) / 'figures', runs_here)
for name, path in sorted(figures.items()):
    print(f'{name:<12} {path}')
print()
print('border radius:', charts.BORDER_RADIUS, 'every rectangle is square by construction')
print()
charts.show(figures)


## 12. The runs as one dataset, and the failures as work

The numbers are written twice over. Once as a flat table with the run's own
counters left in their own columns, so a row can be filtered and joined by
something other than the composite. Once as records that keep the counters
and the checks as objects, because a counter is not a feature until someone
says it is.

On top of the rows sits the part that names the work. Every failed row is
read against a small table of signals, each of which is a measurable fact
about the row rather than an opinion about it, and each of which names the
part of the harness it points at: the output contract, the tool catalog,
the argument schema, the retry rule, the stop condition, the plan, or the
verifier. The mapping is deliberately short. It says where to look, not what
to change, and it is stated as data, so a reader can disagree with one entry
without touching the arithmetic.

The figure that orders the work is the composite a solver did not get, over
the rows a signal fired on. A dimension averaged over the suite hides the
four tasks a solver missed while scoring near a hundred on the five it
passed, and the four tasks are the work.


In [ ]:
reading_here = dataset.learning(runs_here)
print(dataset.render(reading_here), end='')
print()
print('signals:', ', '.join(entry['name'] for entry in dataset.SIGNALS))
print('areas  :', ', '.join(reading_here['priority']))


## 13. Everything the run wrote, in one file, with a link

A hosted session ends and its files with it. A run whose result cannot be
carried off the machine it ran on is not evidence, so the last step writes
every file into one zip and offers a link to it: the runs, the comparison,
the pictures, the dataset, and a manifest naming each member with its size
and a digest, so a reader can tell what they hold without unpacking it.

The zip is written beside the run and the link points at it. Nothing here
reaches the network.


In [ ]:
record = report.build(sides, Path(WORKING_DIR))
print(report.render(record), end='')
print()
charts.show(record['charts'])
print()
print('download:', bundle.show(record['bundle']['path'], 'benchmark-bundle.zip'))


## Notes on what this does not say

* These are fixture solvers, not models. The table measures the suite and
  the arithmetic, and it is the calibration a model's run is read against.
* The dataset is the runs and nothing else: no host, no path outside the run,
  no clock. A row carries the solver label and the side, which is what lets a
  run made later be set beside this one.
* A signal is a reading of counters the loop wrote, so it can be recomputed
  from the CSV by anyone who doubts it. The reading is a place to look, not a
  verdict.
* Latency is measured here and on the machine, and is not comparable across
  them: it is reported as what it is, a property of the runtime.
* Token counts are zero because no model was called. A live profile fills
  them the same way it fills everything else.
* A difference between two sides on the same solver label is a finding about
  portability, and the comparison names the task and both readings.


In [ ]:
print('runs written:'); [print(' ', f) for f in written]
print('charts     :', len(record['charts']))
print('dataset    :', record['dataset']['descriptor'])
print('bundle     :', record['bundle']['path'], record['bundle']['size'], 'bytes')
print('done')
